# 07 · Análisis landmark y de SMM incidente — Morbilidad materna severa MIMIC-IV (M2)

| Campo | Contenido |
|---|---|
| Secuencia de notebooks | 01: E1 · E2 · E3 · E4 · P1 → 02: E5 · P2 → 03: E6 · E7 · P3 · R1 · E8 → 04: E8 → 05: E9 (E7) → 06: E9 (E3) → **07: E9 · C1-C3 · R2 · R4** → 08: E9 · C4 → 09: E9 · C4 → 10: P4 · E10 · E11 |
| Qué hace | Vuelve a correr el pipeline principal en poblaciones y etiquetas que retiran la morbilidad ya presente en T = admisión + 24 h (criterios C1 y C2), prueba el sesgo de detección (C3), una definición alternativa de SMM, predictores ampliados (retorno R4), el escalamiento de la atención después de T (retorno R2), raza y etnia por grupo (E10), la puerta P3 del desenlace de R2 y la reducción por tamaño de muestra |
| Etapas y puertas | E9 · C1 · C2 · C3 · R4 → E7 · R2 → E3 · P3 · E10 |
| Entradas | Base local PostgreSQL de MIMIC-IV (tablas de `sql/`): tablas del notebook 03 más `timing_flags_v3`, `smm_code_hits_v3`, `escalation_v3` (`sql/07_timing_audit/`) |
| Salidas | `data/outputs_<análisis>.txt` y los resúmenes `outputs_richer_predictors_summary.txt`, `outputs_escalation_summary.txt`, `outputs_c3_labs_summary.txt`, `outputs_g3_escalation.txt`, `outputs_escalation_reduced*.txt` |
| Uso de los datos | Solo se muestran y guardan resultados agregados; nada a nivel de paciente. No agregar celdas que muestren o exporten filas |
| Tiempo | Varias horas en total; cada sección se puede correr sola con los interruptores `RUN_...` |
| Siguiente | `08_era_temporal_validation_smm_es.ipynb` |
| Versión en inglés | `07_landmark_incident_smm.ipynb` (mismo código) |

| Clase de momento | Regla (solo admisiones con SMM) |
|---|---|
| `1 prevalent` | un indicador por procedimiento fechado antes de T, **o** ingreso a UCI antes de T, **o** código de falla renal con inicio de lesión renal aguda por laboratorio en T o antes |
| `2 indeterminate` | nada de lo anterior, y un indicador por procedimiento en el día calendario que contiene a T |
| `3 incident (documented)` | nada de lo anterior, y un indicador por procedimiento después de T **o** código de falla renal con inicio de lesión renal aguda por laboratorio después de T |
| `4 untimed` | el resto (solo códigos de diagnóstico, sin momento por laboratorio) |

Las admisiones se retiran; nunca se reetiquetan como no eventos. Correr primero `sql/07_timing_audit/01_timing_flags_v3.sql`.

> **Orden de ejecución.** Correr todas las celdas de arriba hacia abajo. Primero `QUICK = True` como prueba rápida (minutos; nunca se reportan sus cifras) y después `QUICK = False`.

## E1 · Gobierno y configuración — librerías, conexión y configuración

Fija la semilla, los pliegues externos e internos, las repeticiones, las réplicas del bootstrap y los conteos de referencia `EXPECTED` congelados en la puerta P1. `QUICK = True` es una prueba rápida (minutos); sus cifras nunca se reportan.

In [1]:
import os, sys, time, math, warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# notebooks/ -> project root is one level up
sys.path.append(os.path.abspath(os.path.join("..")))
import libraries.database as db_lib
engine = db_lib.get_connect_database()

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

# ----------------------------------------------------------------------------- settings
QUICK        = False         # True = smoke test (small forests, 1 repeat, B = 50). Never report QUICK numbers.
SEED         = 20260101
OUTER_FOLDS  = 5
N_REPEATS    = 3            # repeats of the outer grouped CV (development instability)
INNER_FOLDS  = 3
BOOT         = 1000         # cluster bootstrap replicates for the main analysis
BOOT_SENS    = 300          # replicates for the sensitivity analyses
N_JOBS       = -1
N_TREES      = 300
XGB_TREES    = 200
APPLY_LIMITS = True         # set values outside the plausibility limits of the dictionary to missing
CAT_COLS     = ["ethnicity", "insurance", "language", "admission_type", "admission_location"]
DCA_T        = [0.005, 0.01, 0.02, 0.03, 0.05, 0.10, 0.15, 0.20]
EXPECTED     = {"eligible_deliveries": 19383, "smm": 445, "deaths": 3}   # from the audit counts (Q-F / Q-G)

if QUICK:
    OUTER_FOLDS, N_REPEATS, BOOT, BOOT_SENS = 3, 1, 50, 30
    N_TREES, XGB_TREES = 60, 50
print(f"QUICK={QUICK}  outer={OUTER_FOLDS}x{N_REPEATS}  inner={INNER_FOLDS}  B={BOOT}  trees={N_TREES}")

# ----------------------------------------------------------------------------- language and presentation
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
# Nothing below changes a computation: it only decides how tables and messages are shown. The English aggregate
# reports data/outputs_*.txt are written exactly as before (the article cites them); the Spanish notebook also
# writes a translated copy, data/outputs_*_es.txt.
LANG = "es"
import builtins, contextlib, importlib, io, html as _html
import numpy as np
import libraries.i18n_es, libraries.i18n          # reloaded so that a running kernel picks up dictionary changes
importlib.reload(libraries.i18n_es); importlib.reload(libraries.i18n)
from libraries.i18n import translate
pd.set_option("display.max_colwidth", None)
def _tr(s, lang=None):
    """Translate one line keeping its indentation and trailing spaces."""
    lang = lang or LANG
    if lang != "es" or not isinstance(s, str):
        return s
    out = []
    for ln in s.split("\n"):
        core = ln.strip()
        if not core:
            out.append(ln); continue
        i = ln.index(core)
        out.append(ln[:i] + translate(core, "es") + ln[i + len(core):])
    return "\n".join(out)

_REC = None          # while emit() runs: the blocks of the English report, [("text", line) | ("table", frame, index)]

def print(*args, sep=" ", end="\n", file=None, flush=False):
    """Inside emit() the English text goes to the report unchanged; outside it, messages follow LANG."""
    if _REC is not None and file is None:
        _REC.append(("text", sep.join(str(a) for a in args)))
        return builtins.print(*args, sep=sep, end=end, flush=flush)
    if LANG == "es":
        args = [_tr(a, "es") for a in args]
    return builtins.print(*args, sep=sep, end=end, file=file, flush=flush)

def _cell_fmt(v):
    # Display format of a cell: thousands separator for counts, 4 significant digits for small values.
    if isinstance(v, (bool, np.bool_)) or v is None or (isinstance(v, float) and np.isnan(v)):
        return v
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        if float(v).is_integer() and abs(v) >= 1000:
            return f"{int(v):,}"
        return f"{v:.4g}" if abs(v) < 1 else f"{v:,.4f}".rstrip("0").rstrip(".")
    return v

def _html_table(df, index):
    df = df.astype(object)
    df = df.map(_cell_fmt) if hasattr(df, "map") else df.applymap(_cell_fmt)
    return f"<div style='max-height:420px; overflow:auto'>{df.to_html(index=index, na_rep='', border=0)}</div>"

def _in_notebook():
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except ImportError:
        return False

def table(df, index=False, title=None):
    """Show a table. Inside emit() it also prints the English text of the report (df.to_string)."""
    if isinstance(df, pd.Series):
        df = df.to_frame()
        index = True
    if _REC is not None:
        _REC.append(("table", df, index))
        return builtins.print(df.to_string(index=index))
    if not _in_notebook():
        return builtins.print(translate(df, LANG).to_string(index=index))
    from IPython.display import HTML, display as _display
    head = f"<p style='margin:14px 0 4px 0'><b>{_html.escape(_tr(title))}</b></p>" if title else ""
    _display(HTML(head + _html_table(translate(df, LANG), index)))

def _blocks_html(blocks):
    """Report blocks as HTML: a text line right before a table is its title; rules of '=' are dropped."""
    out, pending = [], []
    def flush_text():
        for ln in pending:
            if ln.strip():
                out.append(f"<div style='white-space:pre-wrap; font-family:monospace; margin:2px 0'>{_html.escape(_tr(ln.rstrip()))}</div>")
        pending.clear()
    for b in blocks:
        if b[0] == "text":
            pending.extend(l for l in b[1].split("\n") if not set(l.strip()) <= {"="} or not l.strip())
        else:
            title = None
            while pending and not pending[-1].strip():
                pending.pop()
            if pending:
                title = pending.pop().strip()
            flush_text()
            if title:
                out.append(f"<p style='margin:14px 0 4px 0'><b>{_html.escape(_tr(title))}</b></p>")
            out.append(_html_table(translate(b[1], LANG), b[2]))
    flush_text()
    return "\n".join(out)

def _blocks_text_es(blocks):
    """Spanish copy of a report (data/outputs_*_es.txt)."""
    lines = []
    for b in blocks:
        if b[0] == "text":
            lines.extend(_tr(l, "es") for l in b[1].split("\n"))
        else:
            lines.extend(translate(b[1], "es").to_string(index=b[2]).split("\n"))
    return "\n".join(lines) + "\n"

def _emit_to(path, fn, show=True):
    """Run fn, write its English text to `path` exactly as printed and, from the Spanish notebook, a translated copy
    (*_es.txt); show the report as tables."""
    global _REC
    buf = io.StringIO()
    _REC = []
    try:
        with contextlib.redirect_stdout(buf):
            fn()
    finally:
        blocks, _REC = _REC, None
    text = buf.getvalue()
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(text)
    if show:
        show_report(blocks)
    if LANG == "es":
        with open(path.replace(".txt", "_es.txt"), "w", encoding="utf-8") as fh:
            fh.write(_blocks_text_es(blocks))
    return text

def show_report(blocks):
    if _in_notebook():
        from IPython.display import HTML, display as _display
        _display(HTML(_blocks_html(blocks)))
    else:
        for b in blocks:
            builtins.print(_tr(b[1]) if b[0] == "text" else translate(b[1], LANG).to_string(index=b[2]))

def notebook_summary(rows, title="Notebook summary"):
    """Summary of the notebook (as in the neonatal repository): one value per row; a value that cannot be computed
    (for example, an analysis that was not run in this session) is shown as n/a."""
    out = {}
    for label, fn in rows:
        try:
            v = fn() if callable(fn) else fn
        except Exception:
            v = "n/a"
        out[label] = v
    table(pd.Series(out, name="value").to_frame(), index=True, title=title)

def _m(res, model, met, d=3):
    """Point value of a metric with its 95% bootstrap interval, from a result with "point" and "boot"."""
    v = f"{res['point'][(model, met)]:.{d}f}"
    b = res.get("boot") or {}
    a = b.get((model, met)) if isinstance(b, dict) else None
    if a is not None and len(a) >= 20:
        lo, hi = np.nanpercentile(a, [2.5, 97.5])
        v += f" ({lo:.{d}f}-{hi:.{d}f})"
    return v

def _cv_auroc(res, model):
    """AUROC averaged over the repeats of a cross-validation result that only keeps out-of-fold predictions."""
    from sklearn.metrics import roc_auc_score
    y = res["y"]
    return float(np.mean([roc_auc_score(y, res["preds"][model][r]) for r in range(len(res["preds"][model]))]))


QUICK=False  outer=5x3  inner=3  B=1000  trees=300


## E7 · Predictores y E4 / P1 — tabla de predictores, diccionario final y conteos de referencia

`predictors_24h_v2` tiene una fila por hospitalización obstétrica con los predictores observados en la ventana (admisión − 24 h, admisión + 24 h). `predictor_dictionary_final_v2` asigna a cada variable su rol (`main model`, `ICU sensitivity only`, `laboratory sensitivity ...`) y sus límites de plausibilidad; los valores fuera de los límites pasan a faltantes. Los datos cargados se comparan con los conteos de referencia; la celda siguiente muestra la decisión de la puerta P1.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Se reproducen los tres conteos | P1 pasa: sigue a los análisis | 19.383 partos, 445 SMM, 3 muertes |
| Algún conteo difiere | Se detiene: revisar las tablas de `sql/` antes de ajustar cualquier modelo | — |

In [2]:
df = pd.read_sql("SELECT * FROM mimiciv.predictors_24h_v2", engine)
dic = pd.read_sql("""SELECT variable, model_role_final, missingness_indicator_main, plausible_low, plausible_high,
                            n_eligible_delivery, coverage_pct
                     FROM mimiciv.predictor_dictionary_final_v2""", engine)

for c in ["eligible_24h", "icu_in_24h", "is_delivery_v2", "smm_any", "died_same_admission"]:
    df[c] = df[c].fillna(False).astype(bool)
df["admittime"] = pd.to_datetime(df["admittime"])

VAR_MAIN = dic.loc[dic.model_role_final == "main model", "variable"].tolist()
VAR_ICU  = dic.loc[dic.model_role_final == "ICU sensitivity only", "variable"].tolist()
VAR_LAB  = dic.loc[dic.model_role_final.str.startswith("laboratory sensitivity"), "variable"].tolist()
IND_MAIN = dic.loc[dic.missingness_indicator_main.fillna(False).astype(bool), "variable"].tolist()

print("Dictionary roles ->", {"main": VAR_MAIN, "ICU only": len(VAR_ICU), "lab sensitivity": len(VAR_LAB),
                              "missingness indicator in main model": IND_MAIN})

# Plausibility limits (proposals pending clinical confirmation): out-of-range values become missing
if APPLY_LIMITS:
    n_set = {}
    for _, r in dic.dropna(subset=["plausible_low", "plausible_high"]).iterrows():
        v = r["variable"]
        if v in df.columns:
            bad = (df[v] < r["plausible_low"]) | (df[v] > r["plausible_high"])
            n_set[v] = int(bad.sum())
            df.loc[bad, v] = np.nan
    print("Values set to missing by plausibility limits (count per variable):", n_set)

# Missingness indicators (computed row by row from the raw value: no information crosses patients)
for v in set(IND_MAIN + VAR_LAB + VAR_ICU):
    df[f"{v}_missing"] = df[v].isna().astype(int)
df["icu_in_24h_bin"] = df["icu_in_24h"].astype(int)
df["y"] = df["smm_any"].astype(int)

# Audit checks against the counts already certified (aggregate only)
elig = df[df.eligible_24h & df.is_delivery_v2]
chk = {"eligible_deliveries": len(elig), "smm": int(elig.smm_any.sum()), "deaths": int(elig.died_same_admission.sum())}
for k, v in chk.items():
    print(f"{k}: {v}  (expected {EXPECTED[k]})  [{'OK' if v == EXPECTED[k] else 'CHECK: differs from the audit counts'}]")
print("Eligible any obstetric admissions:", int(df.eligible_24h.sum()),
      "| SMM:", int(df[df.eligible_24h].smm_any.sum()), "| deaths:", int(df[df.eligible_24h].died_same_admission.sum()))

Roles del diccionario -> {'main': ['hemoglobin', 'creatinine'], 'ICU only': 14, 'lab sensitivity': 11, 'missingness indicator in main model': ['creatinine']}
Valores llevados a faltante por los límites de plausibilidad (conteo por variable): {'hemoglobin': 0, 'hematocrit': 0, 'creatinine': 1}
partos elegibles: 19383  (esperado 19383)  [OK]
SMM: 445  (esperado 445)  [OK]
muertes: 3  (esperado 3)  [OK]
Admisiones obstétricas elegibles de cualquier tipo: 23625 | SMM: 814 | deaths: 5


In [3]:
# Gate G1 decision: the loaded data must reproduce the reference counts frozen at the count audit (aggregates only)
g1 = pd.DataFrame([{"count": k, "loaded": v, "expected": EXPECTED[k],
                    "result": "reproduced" if v == EXPECTED[k] else "DIFFERS"} for k, v in chk.items()])
table(g1, title="Gate G1: reference counts")
table(pd.Series({"G1 decision": "passes: continue to the analyses" if (g1["result"] == "reproduced").all()
                 else "does NOT pass: review the tables of sql/ before fitting any model"}, name="decision"),
      title="Gate G1 decision")

conteo,cargado,esperado,resultado
partos elegibles,"19,383","19,383",reproducido
SMM,445,445,reproducido
muertes,3,3,reproducido


,decisión
decisión P1,pasa: se sigue con los análisis


In [4]:
# Timing classes (one row per eligible delivery hospitalization)
tf = pd.read_sql("""SELECT hadm_id, smm_timing_class, smm_incident_strict, aki_prevalent, aki_incident
                     FROM mimiciv.timing_flags_v3""", engine)
for col in ["smm_incident_strict", "aki_prevalent", "aki_incident"]:
    tf[col] = tf[col].fillna(False).astype(bool)
df = df.drop(columns=[x for x in tf.columns if x != "hadm_id" and x in df.columns]).merge(tf, on="hadm_id", how="left")
df["smm_timing_class"] = df["smm_timing_class"].fillna("not an eligible delivery")
for col in ["smm_incident_strict", "aki_prevalent", "aki_incident"]:
    df[col] = df[col].fillna(False).astype(bool)
df["y_aki"] = df["aki_incident"].astype(int)
n_tf = int((df.eligible_24h & df.is_delivery_v2 & (df.smm_timing_class != "not an eligible delivery")).sum())
print(f"Eligible deliveries with timing flags: {n_tf} (expected {EXPECTED['eligible_deliveries']}) "
      f"[{'OK' if n_tf == EXPECTED['eligible_deliveries'] else 'CHECK'}]")

Partos elegibles con marcas de tiempo: 19383 (esperado 19383) [OK]


## E6 · Punto de predicción y ventana — conjuntos de análisis y de predictores

Cada análisis es una especificación (población, conjunto de predictores, etiqueta). El punto de predicción es la admisión + 24 h; solo se usa la información registrada hasta ese momento.

| Nombre | Población | Predictores |
|---|---|---|
| `main` | Hospitalizaciones de parto elegibles | Variables del modelo principal del diccionario (hemoglobina, creatinina), indicador de creatinina faltante, edad, variables categóricas de la admisión |
| `first_admission` | Primera hospitalización de parto de cada paciente, si es elegible | como `main` |
| `any_obstetric` | Hospitalizaciones obstétricas elegibles (con o sin parto) | como `main` |
| `labs_extended` | como `main` | `main` + hematocrito y los demás laboratorios, cada uno con su indicador de faltante |
| `icu_vars` | como `main` | `main` + las variables solo de UCI (cada una con indicador de faltante) + `icu_in_24h` |
| `label_ge2` | como `main` | como `main`, desenlace = dos o más indicadores de SMM (`smm_ge2`) |
| `no_icu_24h` | Partos elegibles **sin** ingreso a UCI en las primeras 24 h | como `main` |
| `icd10_only` | Partos elegibles cuyos diagnósticos son todos CIE-10-CM (sin diagnósticos CIE-9-CM) | como `main` |

La tabla anterior es la del notebook 03. Este notebook usa `main` (referencia) y las cuatro especificaciones nuevas de la celda siguiente: `landmark`, `incident_documented`, `incident_strict` y `aki_lab_incident`.

In [5]:
def make_spec(name):
    d = df[df.eligible_24h & df.is_delivery_v2].copy()
    num = ["age_at_admission"] + VAR_MAIN
    binc = [f"{v}_missing" for v in IND_MAIN]
    label = "y"
    prevalent_or_indet = d.smm_timing_class.isin(["1 prevalent", "2 indeterminate"])
    if name == "main":
        pass
    elif name in ("landmark", "incident_documented", "incident_strict"):
        d = d[~d.icu_in_24h & ~prevalent_or_indet]
        if name == "incident_documented":
            d = d[~(d.y.astype(bool) & (d.smm_timing_class == "4 untimed"))]
        if name == "incident_strict":
            d = d[~(d.y.astype(bool) & ~d.smm_incident_strict)]
    elif name == "aki_lab_incident":
        d = d[~d.icu_in_24h & ~d.aki_prevalent]
        label = "y_aki"
    else:
        raise ValueError(name)
    num  = [c for c in dict.fromkeys(num) if c in d.columns]
    binc = [c for c in dict.fromkeys(binc) if c in d.columns and d[c].nunique() > 1]
    cat  = [c for c in CAT_COLS if c in d.columns]
    d = d.reset_index(drop=True)
    return dict(name=name, data=d, num=num, bin=binc, cat=cat, label=label)

ANALYSES = ["landmark", "incident_documented", "incident_strict", "aki_lab_incident"]
for nm in ["main"] + ANALYSES:
    sp = make_spec(nm)
    d = sp["data"]
    print(f"{nm:20s} admissions={len(d):6d}  patients={d.subject_id.nunique():6d}  events={int(d[sp['label']].sum()):4d}  "
          f"prevalence={d[sp['label']].mean():.4f}")

main                 admissions= 19383  patients= 13710  events= 445  prevalence=0.0230
landmark             admissions= 19198  patients= 13584  events= 285  prevalence=0.0148
incident_documented  admissions= 19027  patients= 13479  events= 114  prevalence=0.0060
incident_strict      admissions= 18996  patients= 13453  events=  83  prevalence=0.0044
aki_lab_incident     admissions= 19175  patients= 13565  events= 202  prevalence=0.0105


## E4 · Auditoría de conteos — admisiones con SMM por clase de momento

Cuántos eventos de SMM se pueden ubicar en el tiempo respecto a T; estos conteos alimentan el diagrama de flujo de la cohorte del artículo.

In [6]:
e = df[df.eligible_24h & df.is_delivery_v2]
tab = (e[e.y == 1].groupby("smm_timing_class").size().rename("SMM admissions").to_frame())
tab["% of SMM"] = (100 * tab["SMM admissions"] / tab["SMM admissions"].sum()).round(1)
table(tab, index=True)
print(f"\nStrict incident events: {int((e.y.astype(bool) & e.smm_incident_strict).sum())}")
print(f"Laboratory AKI at or before T: {int(e.aki_prevalent.sum())};  after T: {int(e.aki_incident.sum())} "
      f"(of which with an SMM code: {int((e.aki_incident & e.y.astype(bool)).sum())})")
print(f"ICU admission before T: {int(e.icu_in_24h.sum())} admissions ({int((e.icu_in_24h & e.y.astype(bool)).sum())} with SMM)")

,SMM admissions,% de la SMM
smm_timing_class,,
1 prevalent,150,33.7
2 indeterminate,10,2.2
3 incident (documented),114,25.6
4 untimed,171,38.4



Eventos incidentes estrictos: 83
LRA por laboratorio en T o antes: 110;  después de T: 217 (de ellas con código de SMM: 113)
Ingreso a UCI antes de T: 115 admisiones (90 con SMM)


## E8 · Modelado, validación y métricas — preprocesamiento, algoritmos, ajuste anidado, ensamble y calibración

Todo lo que aprende de los datos queda dentro de la parte de entrenamiento de cada partición (puerta P2: ninguna paciente a ambos lados):

* `Winsor` recorta cada columna numérica a los percentiles 0,5–99,5 *de los datos de entrenamiento*.
* La imputación por la mediana, el escalado y la codificación one-hot se ajustan dentro de un `Pipeline`, una vez por ajuste.
* Los hiperparámetros se eligen con **validación cruzada interna agrupada por paciente** (precisión promedio de las predicciones fuera de muestra).
* El meta-modelo del ensamble se entrena con las predicciones **fuera de muestra** de los algoritmos base (nunca con predicciones dentro de muestra).
* El calibrador de Platt se entrena con predicciones fuera de muestra de las pacientes de entrenamiento; las pacientes de prueba externa solo se evalúan.

In [7]:
def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def expit(z):
    return 1.0 / (1.0 + np.exp(-z))

class Winsor(BaseEstimator, TransformerMixin):
    """Clip each column to training percentiles (fit on the training data only)."""
    def __init__(self, lo=0.5, hi=99.5):
        self.lo = lo; self.hi = hi
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        self.lo_ = np.nanpercentile(X, self.lo, axis=0)
        self.hi_ = np.nanpercentile(X, self.hi, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, float), self.lo_, self.hi_)

def make_prep(num, binc, cat):
    parts = [("num", Pipeline([("w", Winsor()),
                               ("imp", SimpleImputer(strategy="median", keep_empty_features=True)),
                               ("sc", StandardScaler())]), num)]
    if binc:
        parts.append(("bin", "passthrough", binc))
    if cat:
        parts.append(("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                                       ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                            min_frequency=0.01, sparse_output=False))]), cat))
    return ColumnTransformer(parts)

def prep_frame(d, num, binc, cat):
    X = d[num + binc + cat].copy()
    for c in cat:
        X[c] = X[c].astype(object).where(X[c].notna(), np.nan)
    return X

# Small prespecified grids (class weights are options; no resampling)
GRID = {
    # The grids cover values beyond those most often chosen in preliminary runs, so that the selected value
    # is not at the edge of the grid (ET depth, RF depth and XGB depth; class weights remain options).
    "LR":  [dict(C=C, cw=cw) for C in (0.01, 0.1, 1.0) for cw in (None, "balanced")],
    "RF":  [dict(depth=dp, cw=cw) for dp in (2, 4, 8) for cw in (None, "balanced_subsample")],
    "ET":  [dict(depth=dp, cw=cw) for dp in (4, 8, 12) for cw in (None, "balanced_subsample")],
    "XGB": [dict(depth=dp, spw=sp) for dp in (1, 2, 4) for sp in (1, "sqrt")],
}
MODELS = ["LR", "RF", "ET", "XGB", "Stack", "Stack_raw"]
BOOT_MODELS = ["LR", "RF", "ET", "XGB", "Stack"]
PAIRS = [("RF", "LR"), ("ET", "LR"), ("XGB", "LR"), ("Stack", "LR"), ("Stack", "XGB")]

def cfg_str(c):
    return ",".join(f"{k}={v}" for k, v in c.items())

def make_clf(name, cfg, ytr, seed):
    if name == "LR":
        return LogisticRegression(C=cfg["C"], class_weight=cfg["cw"], max_iter=3000)
    if name == "RF":
        return RandomForestClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                      class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "ET":
        return ExtraTreesClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                    class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "XGB":
        pos = max(int(ytr.sum()), 1); neg = max(len(ytr) - pos, 1)
        spw = math.sqrt(neg / pos) if cfg["spw"] == "sqrt" else 1.0
        return XGBClassifier(n_estimators=XGB_TREES, learning_rate=0.05, max_depth=cfg["depth"], subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=5, scale_pos_weight=spw, eval_metric="logloss",
                             tree_method="hist", n_jobs=N_JOBS if N_JOBS and N_JOBS > 0 else 0, random_state=seed)
    raise ValueError(name)

def fit_predict(name, cfg, prep, Xtr, ytr, Xte, seed):
    pipe = Pipeline([("prep", clone(prep)), ("clf", make_clf(name, cfg, ytr, seed))])
    pipe.fit(Xtr, ytr)
    return pipe.predict_proba(Xte)[:, 1]

def platt(o, t, ytr):
    c = LogisticRegression(penalty=None, max_iter=1000).fit(logit(o).reshape(-1, 1), ytr)
    return c.predict_proba(logit(t).reshape(-1, 1))[:, 1]

def tune_stack_calibrate(Xtr, ytr, gtr, Xte, prep, seed):
    """Inside one outer training set: tune, build out-of-fold predictions, stack, calibrate; score the outer test set."""
    inner = list(StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed).split(Xtr, ytr, gtr))
    names = list(GRID)
    chosen, oof, pred_te = {}, {}, {}
    for name in names:
        best = None
        for cfg in GRID[name]:
            o = np.zeros(len(ytr))
            for tr, va in inner:
                o[va] = fit_predict(name, cfg, prep, Xtr.iloc[tr], ytr[tr], Xtr.iloc[va], seed)
            ap = average_precision_score(ytr, o)
            if best is None or ap > best[0]:
                best = (ap, cfg, o)
        chosen[name] = best[1]; oof[name] = best[2]
        pred_te[name] = fit_predict(name, best[1], prep, Xtr, ytr, Xte, seed)
    # explicit stacking: logistic meta-learner on out-of-fold base predictions (C = 0.1 as in the original framework)
    Z_oof = np.column_stack([logit(oof[n]) for n in names])
    Z_te = np.column_stack([logit(pred_te[n]) for n in names])
    meta = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof, ytr)
    stack_te_raw = meta.predict_proba(Z_te)[:, 1]
    # cross-fitted meta out-of-fold predictions -> calibrator never sees the predictions of its own training rows
    mo = np.zeros(len(ytr))
    for tr, va in StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed + 1).split(Z_oof, ytr, gtr):
        m = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof[tr], ytr[tr])
        mo[va] = m.predict_proba(Z_oof[va])[:, 1]
    out = {n: platt(oof[n], pred_te[n], ytr) for n in names}
    out["Stack"] = platt(mo, stack_te_raw, ytr)
    out["Stack_raw"] = stack_te_raw
    return out, chosen

def run_cv(spec, label=translate("", LANG)):
    d = spec["data"]
    X = prep_frame(d, spec["num"], spec["bin"], spec["cat"])
    y = d[spec["label"]].values.astype(int)
    g = d["subject_id"].values
    prep = make_prep(spec["num"], spec["bin"], spec["cat"])
    n = len(y)
    preds = {m: np.full((N_REPEATS, n), np.nan) for m in MODELS}
    chosen_log = []
    split_log = []
    t0 = time.time(); done = 0; total = N_REPEATS * OUTER_FOLDS
    for r in range(N_REPEATS):
        outer = StratifiedGroupKFold(OUTER_FOLDS, shuffle=True, random_state=SEED + r)
        for k, (tr, te) in enumerate(outer.split(X, y, g)):
            assert len(set(g[tr]) & set(g[te])) == 0, "patient present in training and test"
            split_log.append(dict(repeat=r + 1, fold=k + 1,
                                  train_adm=len(tr), train_pat=len(set(g[tr])), train_events=int(y[tr].sum()),
                                  train_event_pat=len(set(g[tr][y[tr] == 1])),
                                  test_adm=len(te), test_pat=len(set(g[te])), test_events=int(y[te].sum()),
                                  test_event_pat=len(set(g[te][y[te] == 1])), shared_patients=0))
            out, chosen = tune_stack_calibrate(X.iloc[tr], y[tr], g[tr], X.iloc[te], prep, SEED + 100 * r + k)
            for m in MODELS:
                preds[m][r, te] = out[m]
            for nme, cfg in chosen.items():
                chosen_log.append((nme, cfg_str(cfg)))
            done += 1
            if done == 1:
                el = time.time() - t0
                print(f"[{spec['name']}] first outer fold took {el/60:.1f} min -> projected total about {el*total/60:.0f} min")
    print(f"[{spec['name']}] cross-validation finished in {(time.time()-t0)/60:.1f} min; "
          f"every admission scored exactly once per repeat: {bool(np.isfinite(preds['Stack']).all())}")
    return dict(spec=spec, y=y, g=g, preds=preds, chosen=chosen_log, X=X, prep=prep, splits=pd.DataFrame(split_log))

## E8 · Modelado, validación y métricas — métricas, bootstrap por conglomerados y diferencias pareadas

La discriminación, la calibración y la utilidad clínica se calculan con las predicciones fuera de muestra de cada repetición y se promedian entre repeticiones. El bootstrap remuestrea **pacientes** (todas las admisiones de una paciente se mueven juntas), así que los intervalos reflejan las admisiones repetidas. Las diferencias entre modelos usan las **mismas** muestras bootstrap (pareadas).

In [8]:
def citl(y, z, w):
    """Calibration-in-the-large: intercept with the slope fixed at 1 (weighted Newton)."""
    a = 0.0
    for _ in range(40):
        pr = expit(z + a)
        g = np.sum(w * (y - pr)); h = np.sum(w * pr * (1 - pr))
        if h < 1e-12:
            break
        step = g / h; a += step
        if abs(step) < 1e-9:
            break
    return a

def calib_slope(y, z, w):
    """Calibration slope (weighted logistic recalibration, Newton)."""
    a, b = 0.0, 1.0
    for _ in range(40):
        pr = expit(a + b * z)
        r = w * (y - pr); v = w * pr * (1 - pr)
        gr = np.array([r.sum(), (r * z).sum()])
        H = np.array([[v.sum(), (v * z).sum()], [(v * z).sum(), (v * z * z).sum()]])
        try:
            dlt = np.linalg.solve(H, gr)
        except np.linalg.LinAlgError:
            break
        a += dlt[0]; b += dlt[1]
        if np.abs(dlt).max() < 1e-9:
            break
    return b

def top_threshold(p, w, k):
    order = np.argsort(-p)
    cw = np.cumsum(w[order])
    idx = int(np.searchsorted(cw, k * cw[-1]))
    return p[order][min(idx, len(p) - 1)]

def metrics_one(y, p, w):
    z = logit(p)
    m = {}
    m["auroc"] = roc_auc_score(y, p, sample_weight=w)
    m["auprc"] = average_precision_score(y, p, sample_weight=w)
    m["brier"] = float(np.average((p - y) ** 2, weights=w))
    m["slope"] = calib_slope(y, z, w)
    m["citl"] = citl(y, z, w)
    m["oe"] = float(np.sum(w * y) / np.sum(w * p))
    edges = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
    bins = np.searchsorted(edges, p)
    W = w.sum(); ece = 0.0
    for b in range(10):
        mk = bins == b; wb = w[mk].sum()
        if wb > 0:
            ece += wb / W * abs(np.sum(w[mk] * y[mk]) / wb - np.sum(w[mk] * p[mk]) / wb)
    m["ece"] = ece
    ev = np.sum(w * y)
    for k in (0.05, 0.10):
        sel = p >= top_threshold(p, w, k)
        m[f"sens_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / ev
        m[f"ppv_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / np.sum(w[sel])
    for pt in DCA_T:
        sel = p >= pt
        tp = np.sum(w[sel] * y[sel]); fp = np.sum(w[sel] * (1 - y[sel]))
        m[f"nb_{pt}"] = tp / W - fp / W * pt / (1 - pt)
    return m

def all_metrics(y, preds, w, models, strata):
    """Metrics for each model, averaged over repeats. Returns dict[(model, metric)] and the per-repeat list."""
    per = []
    for r in range(N_REPEATS):
        cur = {}
        for mname in models:
            for k, v in metrics_one(y, preds[mname][r], w).items():
                cur[(mname, k)] = v
        prev = np.sum(w * y) / w.sum()
        for pt in DCA_T:
            cur[("TreatAll", f"nb_{pt}")] = prev - (1 - prev) * pt / (1 - pt)
        for sname, mask in strata.items():
            for mname in ("LR", "Stack"):
                if mname not in models:
                    continue
                wm = w[mask]; ym = y[mask]; pm = preds[mname][r][mask]
                if np.sum(wm * ym) >= 1 and np.sum(wm * (1 - ym)) >= 1:
                    cur[(mname, f"auroc_{sname}")] = roc_auc_score(ym, pm, sample_weight=wm)
                    cur[(mname, f"auprc_{sname}")] = average_precision_score(ym, pm, sample_weight=wm)
                else:
                    cur[(mname, f"auroc_{sname}")] = np.nan
                    cur[(mname, f"auprc_{sname}")] = np.nan
        per.append(cur)
    keys = per[0].keys()
    mean = {k: float(np.nanmean([c[k] for c in per])) if not all(np.isnan(c[k]) for c in per) else np.nan for k in keys}
    return mean, per

def bootstrap(res, B, strata, seed):
    y, g, preds = res["y"], res["g"], res["preds"]
    ug, inv = np.unique(g, return_inverse=True)
    G = len(ug)
    rng = np.random.default_rng(seed)
    out = {}
    t0 = time.time()
    for b in range(B):
        cnt = np.bincount(rng.integers(0, G, G), minlength=G)
        w = cnt[inv].astype(float)
        if np.sum(w * y) < 1:
            continue
        mean, _ = all_metrics(y.astype(float), preds, w, BOOT_MODELS, strata)
        for k, v in mean.items():
            out.setdefault(k, []).append(v)
        for a_, b_ in PAIRS:            # paired differences (same bootstrap sample)
            for met in ("auroc", "auprc", "brier") + tuple(f"nb_{pt}" for pt in DCA_T):
                out.setdefault((f"{a_}-{b_}", met), []).append(mean[(a_, met)] - mean[(b_, met)])
        if b == 4:
            el = time.time() - t0
            print(f"  bootstrap: 5 replicates took {el:.0f} s -> projected {el*B/5/60:.1f} min for B={B}")
    return {k: np.array(v) for k, v in out.items()}

## E8 · E11 Reporte — función de reporte

Arma el reporte agregado de un análisis: discriminación, calibración, diferencias pareadas entre modelos, eventos captados en el 5% / 10% de mayor riesgo, beneficio neto, inestabilidad entre repeticiones y los hiperparámetros elegidos en los ciclos internos. `emit()` lo guarda como `data/outputs_<análisis>.txt` (en inglés, como lo cita el artículo) y lo muestra como tablas; el notebook en español también escribe `data/outputs_<análisis>_es.txt`.

In [9]:
def _ci(boot, key, d=3):
    a = boot.get(key)
    if a is None or len(a) < 20:
        return ""
    lo, hi = np.nanpercentile(a, [2.5, 97.5])
    return f" ({lo:.{d}f}-{hi:.{d}f})"

def decile_table(res, models=("LR", "Stack")):
    y = res["y"].astype(float)
    out = None
    for m in models:
        acc = []
        for r in range(N_REPEATS):
            p = res["preds"][m][r]
            edges = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
            b = np.searchsorted(edges, p)
            acc.append(pd.DataFrame({"b": b, "p": p, "y": y}).groupby("b").agg(n=("y", "size"), pred=("p", "mean"), obs=("y", "mean")))
        mean = sum(acc) / len(acc)
        t = pd.DataFrame({"decile": mean.index + 1, "n": mean["n"].round(0).astype(int)}) if out is None else out
        t[f"{m}_pred"] = mean["pred"].round(4).values
        t[f"{m}_obs"] = mean["obs"].round(4).values
        out = t
    return out

def plot_calibration(res):
    import matplotlib.pyplot as plt
    t = decile_table(res)
    fig, ax = plt.subplots(figsize=(5, 5))
    mx = float(max(t["LR_pred"].max(), t["LR_obs"].max(), t["Stack_pred"].max(), t["Stack_obs"].max())) * 1.1
    ax.plot([0, mx], [0, mx], color="grey", lw=1, ls="--", label=translate("ideal", LANG))
    ax.plot(t["LR_pred"], t["LR_obs"], "o-", label=translate("Logistic regression", LANG))
    ax.plot(t["Stack_pred"], t["Stack_obs"], "s-", label=translate("Stacked ensemble", LANG))
    ax.set_xlabel(translate("Mean predicted risk (decile)", LANG)); ax.set_ylabel(translate("Observed SMM rate", LANG))
    ax.set_title(translate(f"Calibration by decile -- {res['spec']['name']}", LANG)); ax.legend(); plt.show()   # displayed only, not saved

def emit(name, fn):
    """Write the aggregate report data/outputs_<name>.txt (English text, as cited by the article) and show it as tables;
    the Spanish notebook also writes data/outputs_<name>_es.txt. Aggregates only."""
    path = f"../data/outputs_{name}.txt"
    text = _emit_to(path, fn)
    print(f"[aggregate report saved as data/{os.path.basename(path)}  ({len(text)} characters)]")

def run_analysis(name, B):
    spec = make_spec(name)
    res = run_cv(spec)
    d = spec["data"]
    strata = {}
    if name == "main":
        strata = {"icu1": d["icu_in_24h"].values.astype(bool), "icu0": ~d["icu_in_24h"].values.astype(bool)}
    y = res["y"].astype(float)
    res["point"], res["per"] = all_metrics(y, res["preds"], np.ones(len(y)), MODELS + [], strata)
    res["boot"] = bootstrap(res, B, strata, SEED + 7)
    res["strata"] = strata
    return res

def report(res):
    sp = res["spec"]; d = sp["data"]; y = res["y"]
    pt, per, bt = res["point"], res["per"], res["boot"]
    n, ev, npat = len(y), int(y.sum()), d.subject_id.nunique()
    print("=" * 110)
    print(f"ANALYSIS {sp['name']}: {n} admissions, {npat} patients, {ev} events ({100*ev/n:.2f}%), "
          f"numeric={len(sp['num'])}, binary={len(sp['bin'])}, categorical={len(sp['cat'])}")
    print(f"Repeats={N_REPEATS}, outer folds={OUTER_FOLDS}, inner folds={INNER_FOLDS}, bootstrap B={len(next(iter(bt.values()))) if bt else 0}, QUICK={QUICK}")
    print("=" * 110)

    sl = res["splits"]
    print("\n[0] Patients per outer partition (repeat 1 shown; every split verified to share no patient between training and test)")
    table(sl[sl.repeat == 1].drop(columns=["repeat"]))
    print(f"    Across all {len(sl)} splits: test patients per fold {sl.test_pat.min()}-{sl.test_pat.max()}, "
          f"event-bearing test patients per fold {sl.test_event_pat.min()}-{sl.test_event_pat.max()}, "
          f"patients present on both sides: {int(sl.shared_patients.sum())}")

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        r["sd_auroc_repeats"] = f"{np.std([c[(m,'auroc')] for c in per], ddof=1):.4f}" if len(per) > 1 else "n/a"
        r["sd_auprc_repeats"] = f"{np.std([c[(m,'auprc')] for c in per], ddof=1):.4f}" if len(per) > 1 else "n/a"
        rows.append(r)
    print("\n[1] Discrimination and overall accuracy (mean over repeats; 95% cluster-bootstrap CI)")
    print(f"    prevalence = {y.mean():.4f}  (AUPRC of a non-informative model equals the prevalence)")
    table(pd.DataFrame(rows))

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("slope", 2), ("citl", 2), ("oe", 2), ("ece", 4)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        rows.append(r)
    print("\n[2] Calibration (slope ideal 1; calibration-in-the-large ideal 0; O/E ideal 1; ECE with 10 quantile bins)")
    table(pd.DataFrame(rows))

    rows = []
    for a_, b_ in PAIRS:
        r = {"comparison (first minus second)": f"{a_} - {b_}"}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(a_, met)] - pt[(b_, met)]:+.{dd}f}" + _ci(bt, (f"{a_}-{b_}", met), dd)
        rows.append(r)
    print("\n[3] Paired differences (same bootstrap samples); the last row compares the ensemble with XGBoost")
    table(pd.DataFrame(rows))

    rows = []
    for m in BOOT_MODELS:
        rows.append({"model": m,
                     "sens_top5%": f"{pt[(m,'sens_top5')]:.3f}" + _ci(bt, (m, "sens_top5")),
                     "ppv_top5%": f"{pt[(m,'ppv_top5')]:.3f}",
                     "sens_top10%": f"{pt[(m,'sens_top10')]:.3f}" + _ci(bt, (m, "sens_top10")),
                     "ppv_top10%": f"{pt[(m,'ppv_top10')]:.3f}"})
    print("\n[4] Events captured among the 5% / 10% of admissions with the highest predicted risk")
    table(pd.DataFrame(rows))

    rows = []
    for m in BOOT_MODELS + ["TreatAll"]:
        rows.append({"strategy": m, **{f"pt={t:g}": round(pt[(m, f'nb_{t}')], 5) for t in DCA_T}})
    rows.append({"strategy": "TreatNone", **{f"pt={t:g}": 0.0 for t in DCA_T}})
    print("\n[5] Net benefit (decision curve) at the stated threshold probabilities")
    table(pd.DataFrame(rows))
    rows = []
    for a_, b_ in [("Stack", "LR"), ("RF", "LR"), ("ET", "LR"), ("XGB", "LR"), ("Stack", "XGB")]:
        rows.append({"comparison": f"{a_} - {b_}", **{f"pt={t:g}": f"{pt[(a_,f'nb_{t}')]-pt[(b_,f'nb_{t}')]:+.5f}" + _ci(bt, (f"{a_}-{b_}", f"nb_{t}"), 5) for t in DCA_T[:6]}})
    print("    Paired net-benefit differences:")
    table(pd.DataFrame(rows))

    if res["strata"]:
        rows = []
        for sname, lab in [("icu1", "ICU admission within 24 h"), ("icu0", "no ICU within 24 h")]:
            mk = res["strata"][sname]
            r = {"stratum": lab, "n": int(mk.sum()), "events": int(y[mk].sum())}
            for m in ("LR", "Stack"):
                r[f"{m}_auroc"] = (f"{pt[(m, 'auroc_' + sname)]:.3f}" + _ci(bt, (m, "auroc_" + sname))) if not np.isnan(pt[(m, 'auroc_' + sname)]) else "n/a"
                r[f"{m}_auprc"] = (f"{pt[(m, 'auprc_' + sname)]:.3f}") if not np.isnan(pt[(m, 'auprc_' + sname)]) else "n/a"
            rows.append(r)
        print("\n[6] Performance by ICU admission within the first 24 h (same predictions, stratified evaluation)")
        table(pd.DataFrame(rows))

    print("\n[8] Calibration by decile of predicted risk (outer-test predictions; mean over repeats)")
    table(decile_table(res))

    ch = pd.DataFrame(res["chosen"], columns=["learner", "config"])
    tab = ch.value_counts().rename("outer_fits").reset_index().sort_values(["learner", "outer_fits"], ascending=[True, False])
    print(f"\n[7] Hyperparameters selected in the inner loops (out of {N_REPEATS*OUTER_FOLDS} outer fits per learner)")
    table(tab)

## E9 · C1 · C2 — poblaciones landmark e incidentes

`landmark` sirve al criterio C1; `incident_documented`, `incident_strict` y `aki_lab_incident` sirven a C2. Los análisis con menos de 40 eventos se corren pero se marcan como inestables. `RUN_MAIN_ANALYSES = False` los omite.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Caída del AUROC ≤ 0,05 en la población landmark (C1) | C1 cumple | Se decide sobre el modelo ajustado en el notebook 10 |
| AUROC ≥ 0,75 con límite inferior ≥ 0,70 para un desenlace con tiempo (C2) | C2 cumple | Se decide en el notebook 10 |

In [10]:

RUN_MAIN_ANALYSES = True   # set to False to run only section 8 without repeating the four analyses above
RUN_MAIN_ANALYSES = False 
results = {}
if RUN_MAIN_ANALYSES:
    for nm in ANALYSES:
        sp = make_spec(nm)
        ev = int(sp["data"][sp["label"]].sum())
        if ev < 40:
            print(f"WARNING: {nm} has only {ev} events; estimates will be unstable (report as exploratory).")
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

## E9 · C3 — sesgo de detección de la lesión renal aguda por laboratorio y variable compartida

La lesión renal aguda por laboratorio después de T solo se detecta si se mide la creatinina después de T, y el predictor (creatinina hasta las 24 h) y el desenlace (creatinina posterior) son la misma variable.

| Análisis | Población | Predictores |
|---|---|---|
| `aki_lab_incident_tested` | `aki_lab_incident` restringida a admisiones **con creatinina medida en las primeras 24 h** | como `main` |
| `aki_lab_incident_tested_nocreat` | la misma | como `main` **sin el valor de creatinina** |

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| El AUROC cae 0,05 o menos entre las examinadas | C3 cumple | — |
| El AUROC cae más de 0,05 | C3 no cumple: sesgo de detección | 0,850 en todas, 0,628 entre las examinadas |

In [11]:
CREAT = [v for v in VAR_MAIN if "creat" in v.lower()]
assert len(CREAT) == 1, f"creatinine variable not found in VAR_MAIN: {VAR_MAIN}"
CREAT = CREAT[0]
_make_spec_base = make_spec

def make_spec(name):
    if name not in ("aki_lab_incident_tested", "aki_lab_incident_tested_nocreat"):
        return _make_spec_base(name)
    sp = _make_spec_base("aki_lab_incident")
    d = sp["data"]
    d = d[d[CREAT].notna()].reset_index(drop=True)
    num = [v for v in sp["num"] if not (name.endswith("_nocreat") and v == CREAT)]
    binc = [b for b in sp["bin"] if b in d.columns and d[b].nunique() > 1]
    return dict(name=name, data=d, num=num, bin=binc, cat=sp["cat"], label=sp["label"])

SENS_AKI = ["aki_lab_incident_tested", "aki_lab_incident_tested_nocreat"]
base = _make_spec_base("aki_lab_incident")["data"]
print(f"aki_lab_incident population: {len(base)} admissions, {int(base.y_aki.sum())} events "
      f"({100 * base.y_aki.mean():.2f}%)")
tested = base[CREAT].notna()
print(f"  creatinine measured in the first 24 h: {int(tested.sum())} admissions ({100 * tested.mean():.1f}%), "
      f"{int(base.loc[tested, 'y_aki'].sum())} events ({100 * base.loc[tested, 'y_aki'].mean():.2f}%)")
print(f"  not measured:                          {int((~tested).sum())} admissions, "
      f"{int(base.loc[~tested, 'y_aki'].sum())} events ({100 * base.loc[~tested, 'y_aki'].mean():.2f}%)")
for nm in SENS_AKI:
    sp = make_spec(nm)
    print(f"{nm:34s} admissions={len(sp['data'])}  events={int(sp['data'][sp['label']].sum())}  numeric={sp['num']}  binary={sp['bin']}")

RUN_SECTION_8 = True   # set to False to skip these two analyses when running section 9 only
if RUN_SECTION_8:
    for nm in SENS_AKI:
        ev = int(make_spec(nm)["data"]["y_aki"].sum())
        if ev < 40:
            print(f"WARNING: {nm} has only {ev} events; estimates will be unstable (report as exploratory).")
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

Población aki_lab_incident: 19175 admisiones, 202 eventos (1.05%)
  creatinina medida en las primeras 24 h: 3767 admisiones (19.6%), 165 eventos (4.38%)
  no medida: 15408 admisiones, 37 eventos (0.24%)
aki_lab_incident_tested            admissions=3767  events=165  numeric=['age_at_admission', 'hemoglobin', 'creatinine']  binary=[]
aki_lab_incident_tested_nocreat    admissions=3767  events=165  numeric=['age_at_admission', 'hemoglobin']  binary=[]
[aki_lab_incident_tested] el primer pliegue externo tardó 0.8 min -> total proyectado de unos 12 min
[aki_lab_incident_tested] validación cruzada terminada en 16.5 min; cada admisión evaluada exactamente una vez por repetición: Sí
  bootstrap: 5 réplicas tardaron 2 s -> proyectado 1.8 min para B=300


[reporte agregado guardado como data/outputs_aki_lab_incident_tested.txt  (7797 caracteres)]
[aki_lab_incident_tested_nocreat] el primer pliegue externo tardó 1.9 min -> total proyectado de unos 29 min
[aki_lab_incident_tested_nocreat] validación cruzada terminada en 13.9 min; cada admisión evaluada exactamente una vez por repetición: Sí
  bootstrap: 5 réplicas tardaron 0 s -> proyectado 0.3 min para B=300


[reporte agregado guardado como data/outputs_aki_lab_incident_tested_nocreat.txt  (7844 caracteres)]


## E9 · E3 — definición de SMM sin las admisiones que son SMM solo por condiciones no agudas

La lista de los CDC incluye algunos códigos de condiciones crónicas o incidentales (aneurisma cerebral no roto, estenosis carotídea, amaurosis fugaz, aneurisma aórtico torácico sin ruptura). La auditoría de códigos (`sql/07_timing_audit/02_phenotype_audit_v3.sql`, consulta A2) encontró 14 admisiones con SMM cuyos únicos códigos son de este tipo; se retiran (no se reetiquetan) en `excl_nonacute` y `landmark_excl_nonacute`. Requiere `mimiciv.smm_code_hits_v3`. `RUN_SECTION_9 = False` omite esta sección.

In [12]:
from sqlalchemy import text
NON_ACUTE_CODES = ("I671", "4373", "I6521", "I6522", "G453", "I712")
codes_sql = ", ".join(f"'{x}'" for x in NON_ACUTE_CODES)   # fixed constants, not user input
SQL_NON_ACUTE = text(f"""
    SELECT hadm_id
    FROM mimiciv.smm_code_hits_v3
    GROUP BY hadm_id
    HAVING BOOL_AND(icd_code IN ({codes_sql}))
""")
non_acute = set(pd.read_sql(SQL_NON_ACUTE, engine).hadm_id)   # kept in memory only

if "_make_spec_s8" not in globals():   # wrap only once, so re-running the cell does not recurse
    _make_spec_s8 = make_spec
def make_spec(name):
    if name not in ("excl_nonacute", "landmark_excl_nonacute"):
        return _make_spec_s8(name)
    sp = _make_spec_s8("main" if name == "excl_nonacute" else "landmark")
    d = sp["data"]
    d = d[~d.hadm_id.isin(non_acute)].reset_index(drop=True)
    return dict(name=name, data=d, num=sp["num"], bin=sp["bin"], cat=sp["cat"], label=sp["label"])

SENS_NONACUTE = ["excl_nonacute", "landmark_excl_nonacute"]
elig = df[df.eligible_24h & df.is_delivery_v2]
print(f"Admissions that are SMM only because of non-acute codes: {len(non_acute)} (expected 14 from query A2)  "
      f"[{'OK' if len(non_acute) == 14 else 'CHECK'}]")
print(f"  of which eligible SMM deliveries: {int(elig[elig.hadm_id.isin(non_acute)].y.sum())}")
for ref, nm in (("main", "excl_nonacute"), ("landmark", "landmark_excl_nonacute")):
    a, b = make_spec(ref)["data"], make_spec(nm)["data"]
    print(f"{nm:24s} admissions={len(b)} (reference {ref}: {len(a)})  events={int(b.y.sum())} (reference: {int(a.y.sum())})")

RUN_SECTION_9 = True
if RUN_SECTION_9:
    for nm in SENS_NONACUTE:
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

Admisiones con SMM solo por códigos no agudos: 14 (esperadas 14 según la consulta A2)  [OK]
  de ellas, partos elegibles con SMM: 14
excl_nonacute            admissions=19369 (reference main: 19383)  events=431 (reference: 445)
landmark_excl_nonacute   admissions=19184 (reference landmark: 19198)  events=271 (reference: 285)
[excl_nonacute] el primer pliegue externo tardó 1.2 min -> total proyectado de unos 18 min
[excl_nonacute] validación cruzada terminada en 15.5 min; cada admisión evaluada exactamente una vez por repetición: Sí
  bootstrap: 5 réplicas tardaron 2 s -> proyectado 1.7 min para B=300


[reporte agregado guardado como data/outputs_excl_nonacute.txt  (7391 caracteres)]
[landmark_excl_nonacute] el primer pliegue externo tardó 0.7 min -> total proyectado de unos 10 min
[landmark_excl_nonacute] validación cruzada terminada en 10.3 min; cada admisión evaluada exactamente una vez por repetición: Sí
  bootstrap: 5 réplicas tardaron 2 s -> proyectado 1.6 min para B=300


[reporte agregado guardado como data/outputs_landmark_excl_nonacute.txt  (7743 caracteres)]


## R4 → E7 · C1 · C2 — ¿predictores más ricos mejoran la predicción de la SMM que aún no está presente a las 24 h?

Con la etiqueta completa, los demás laboratorios subieron el AUROC de 0,79 a 0,82; esta sección repite la comparación donde se retiró la SMM presente en T. `landmark_labs_noind` separa la información de los **valores** de la información de **qué exámenes se pidieron** (los indicadores).

| Análisis | Población | Predictores |
|---|---|---|
| `landmark_labs` | `landmark` | principales + todos los laboratorios `laboratory sensitivity`, cada uno con indicador de faltante |
| `landmark_labs_noind` | `landmark` | como `landmark_labs` sin indicadores de faltante |
| `incident_documented_labs` | `incident_documented` | como `landmark_labs` |
| `incident_strict_labs` | `incident_strict` | como `landmark_labs` |

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| La ganancia se mantiene en la población landmark y sin indicadores | R4 podría convertir el modelo en predictor: reclasificar en P4 | — |
| La ganancia viene de qué exámenes se pidieron | El modelo sigue como identificador temprano | Ganancia sobre todo por los indicadores |

In [13]:
if "_make_spec_s9" not in globals():   # wrap only once, so re-running the cell does not recurse
    _make_spec_s9 = make_spec
LAB10 = ["landmark_labs", "landmark_labs_noind", "incident_documented_labs", "incident_strict_labs"]

def make_spec(name):
    if name not in LAB10:
        return _make_spec_s9(name)
    base_name = name.replace("_labs_noind", "").replace("_labs", "")
    sp = _make_spec_s9(base_name)
    d = sp["data"]
    num = list(dict.fromkeys(sp["num"] + [v for v in VAR_LAB if v in d.columns]))
    if name.endswith("_noind"):
        binc = []
    else:
        binc = list(dict.fromkeys(sp["bin"] + [f"{v}_missing" for v in VAR_LAB if f"{v}_missing" in d.columns]))
        binc = [b for b in binc if d[b].nunique() > 1]
    return dict(name=name, data=d, num=num, bin=binc, cat=sp["cat"], label=sp["label"])

print("Laboratory variables added:", VAR_LAB)
for nm in LAB10:
    sp = make_spec(nm)
    print(f"{nm:26s} admissions={len(sp['data'])}  events={int(sp['data'][sp['label']].sum())}  "
          f"numeric={len(sp['num'])}  binary={len(sp['bin'])}  categorical={len(sp['cat'])}")

RUN_SECTION_10 = True
if RUN_SECTION_10:
    for nm in LAB10:
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

def richer_summary():
    rows = []
    for nm in LAB10:
        if nm not in results:
            continue
        pt, bt, y = results[nm]["point"], results[nm]["boot"], results[nm]["y"]
        for m in ("LR", "XGB", "Stack"):
            rows.append({"analysis": nm, "model": m, "events": int(y.sum()), "prevalence": f"{y.mean():.4f}",
                         "auroc": f"{pt[(m, 'auroc')]:.3f}" + _ci(bt, (m, "auroc")),
                         "auprc": f"{pt[(m, 'auprc')]:.3f}" + _ci(bt, (m, "auprc")),
                         "slope": f"{pt[(m, 'slope')]:.2f}", "O/E": f"{pt[(m, 'oe')]:.2f}"})
    print(f"RICHER PREDICTORS SUMMARY (QUICK={QUICK}); references: outputs_landmark, outputs_incident_documented, outputs_incident_strict")
    table(pd.DataFrame(rows))

if RUN_SECTION_10:
    emit("richer_predictors_summary", richer_summary)

Variables de laboratorio agregadas: ['hematocrit', 'sodium', 'anion_gap', 'magnesium', 'chloride', 'lactate', 'phosphorus', 'glucose', 'bicarbonate', 'calcium_noionized', 'potassium']
landmark_labs              admissions=19198  events=285  numeric=14  binary=12  categorical=5
landmark_labs_noind        admissions=19198  events=285  numeric=14  binary=0  categorical=5
incident_documented_labs   admissions=19027  events=114  numeric=14  binary=12  categorical=5
incident_strict_labs       admissions=18996  events=83  numeric=14  binary=12  categorical=5
[landmark_labs] el primer pliegue externo tardó 0.9 min -> total proyectado de unos 14 min
[landmark_labs] validación cruzada terminada en 13.3 min; cada admisión evaluada exactamente una vez por repetición: Sí
  bootstrap: 5 réplicas tardaron 1 s -> proyectado 1.4 min para B=300


[reporte agregado guardado como data/outputs_landmark_labs.txt  (7792 caracteres)]
[landmark_labs_noind] el primer pliegue externo tardó 1.2 min -> total proyectado de unos 18 min
[landmark_labs_noind] validación cruzada terminada en 14.9 min; cada admisión evaluada exactamente una vez por repetición: Sí
  bootstrap: 5 réplicas tardaron 2 s -> proyectado 2.2 min para B=300


[reporte agregado guardado como data/outputs_landmark_labs_noind.txt  (7691 caracteres)]
[incident_documented_labs] el primer pliegue externo tardó 1.4 min -> total proyectado de unos 21 min
[incident_documented_labs] validación cruzada terminada en 15.7 min; cada admisión evaluada exactamente una vez por repetición: Sí
  bootstrap: 5 réplicas tardaron 2 s -> proyectado 1.9 min para B=300


[reporte agregado guardado como data/outputs_incident_documented_labs.txt  (7747 caracteres)]
[incident_strict_labs] el primer pliegue externo tardó 1.4 min -> total proyectado de unos 21 min
[incident_strict_labs] validación cruzada terminada en 16.4 min; cada admisión evaluada exactamente una vez por repetición: Sí
  bootstrap: 5 réplicas tardaron 1 s -> proyectado 1.0 min para B=300


[reporte agregado guardado como data/outputs_incident_strict_labs.txt  (8020 caracteres)]


análisis,modelo,eventos,prevalencia,AUROC,AUPRC,pendiente,O/E
landmark_labs,LR,285,0.0148,0.732 (0.695-0.762),0.061 (0.048-0.080),1.00,0.99
landmark_labs,XGB,285,0.0148,0.737 (0.703-0.769),0.067 (0.053-0.087),1.03,1.00
landmark_labs,Stack,285,0.0148,0.738 (0.702-0.768),0.066 (0.052-0.087),1.01,0.99
landmark_labs_noind,LR,285,0.0148,0.663 (0.627-0.697),0.046 (0.037-0.062),1.03,0.99
landmark_labs_noind,XGB,285,0.0148,0.726 (0.693-0.758),0.065 (0.051-0.084),1.02,0.99
landmark_labs_noind,Stack,285,0.0148,0.724 (0.690-0.757),0.065 (0.052-0.087),1.01,0.99
incident_documented_labs,LR,114,0.0060,0.717 (0.670-0.763),0.024 (0.017-0.041),1.00,0.99
incident_documented_labs,XGB,114,0.0060,0.748 (0.702-0.791),0.025 (0.018-0.038),1.01,0.98
incident_documented_labs,Stack,114,0.0060,0.754 (0.704-0.797),0.027 (0.020-0.046),1.06,0.99
incident_strict_labs,LR,83,0.0044,0.742 (0.681-0.798),0.019 (0.013-0.034),0.95,0.99


[reporte agregado guardado como data/outputs_richer_predictors_summary.txt  (1437 caracteres)]


## R2 → E3 · C2 — escalamiento de la atención después del punto de predicción

Un desenlace con tiempo que no depende de exámenes de laboratorio: ingreso a UCI, inicio de ventilación invasiva, un indicador de SMM por procedimiento o muerte hospitalaria **después de T** (`sql/07_timing_audit/04_escalation_label.sql`, tabla `mimiciv.escalation_v3`). Se excluyen las admisiones con UCI antes de T (esperado: 19.268 admisiones, 101 eventos).

| Análisis | Población | Predictores |
|---|---|---|
| `escalation` | partos elegibles sin UCI antes de T | modelo principal |
| `escalation_labs` | la misma | principales + todos los laboratorios `laboratory sensitivity` con indicadores (R2 combinado con R4) |
| `escalation_landmark` | como `escalation`, también sin SMM presente o indeterminada en T | modelo principal |

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| AUROC ≥ 0,75 y límite inferior del 95% ≥ 0,70 | C2 cumple para un desenlace con tiempo; revisar P3 (sección 14) | 0,763 (0,709–0,808); landmark 0,765 |
| Menos de 100 eventos | Reportar como exploratorio | 101 eventos |

In [14]:
EXPECTED_ESC = {"population": 19268, "escalation": 101}   # Q-E1 of sql/07_timing_audit/03_escalation_counts.sql
esc = pd.read_sql("""SELECT hadm_id, icu_after, vent_after, smm_proc_after, death_after, escalation_any
                     FROM mimiciv.escalation_v3""", engine)
for c in ["icu_after", "vent_after", "smm_proc_after", "death_after", "escalation_any"]:
    esc[c] = esc[c].fillna(False).astype(bool)
chk = {"population": len(esc), "escalation": int(esc.escalation_any.sum())}
for k, v in chk.items():
    print(f"escalation {k}: {v}  (expected {EXPECTED_ESC[k]})  [{'OK' if v == EXPECTED_ESC[k] else 'CHECK: differs from Q-E1'}]")

df = df.drop(columns=[c for c in esc.columns if c != "hadm_id" and c in df.columns])
df = df.merge(esc, on="hadm_id", how="left")
df["in_esc_pop"] = df["escalation_any"].notna()
df["y_esc"] = df["escalation_any"].fillna(False).astype(int)

def _n(x):
    return str(int(x)) if x >= 10 else "<10"

if "_make_spec_s10b" not in globals():   # wrap only once, so re-running the cell does not recurse
    _make_spec_s10b = make_spec
ESC11 = ["escalation", "escalation_labs", "escalation_landmark"]

def make_spec(name):
    if name not in ESC11:
        return _make_spec_s10b(name)
    sp = _make_spec_s10b("main")
    d = sp["data"]
    d = d[d.in_esc_pop & ~d.icu_in_24h]
    if name == "escalation_landmark":
        d = d[~d.smm_timing_class.isin(["1 prevalent", "2 indeterminate"])]
    d = d.reset_index(drop=True)
    num, binc = list(sp["num"]), list(sp["bin"])
    if name == "escalation_labs":
        num = list(dict.fromkeys(num + [v for v in VAR_LAB if v in d.columns]))
        binc = list(dict.fromkeys(binc + [f"{v}_missing" for v in VAR_LAB if f"{v}_missing" in d.columns]))
    binc = [b for b in binc if b in d.columns and d[b].nunique() > 1]
    return dict(name=name, data=d, num=num, bin=binc, cat=sp["cat"], label=translate("y_esc", LANG))

for nm in ESC11:
    sp = make_spec(nm); d = sp["data"]
    print(f"{nm:20s} admissions={len(d):6d}  patients={d.subject_id.nunique():6d}  events={int(d.y_esc.sum()):4d}  "
          f"numeric={len(sp['num'])}  binary={len(sp['bin'])}  categorical={len(sp['cat'])}")

RUN_SECTION_11 = True
RUN_SECTION_11 = False

if RUN_SECTION_11:
    for nm in ESC11:
        ev = int(make_spec(nm)["data"]["y_esc"].sum())
        if ev < 100:
            print(f"NOTE: {nm} has {ev} events (< 100): report as exploratory.")
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

def escalation_summary():
    print(f"ESCALATION AFTER T (RETURN R2) SUMMARY (QUICK={QUICK}); bootstrap B={BOOT_SENS}; counts below 10 shown as <10")
    print("\n[1] Components of the composite (an admission can have several)")
    rows = []
    for nm in ESC11:
        d = make_spec(nm)["data"]; ev = d[d.y_esc == 1]
        rows.append({"analysis": nm, "admissions": _n(len(d)), "escalation": _n(len(ev)),
                     "icu_after": _n(ev.icu_after.sum()), "vent_after": _n(ev.vent_after.sum()),
                     "smm_proc_after": _n(ev.smm_proc_after.sum()), "death_after": _n(ev.death_after.sum()),
                     "with_smm_label": _n(ev.y.sum())})
    table(pd.DataFrame(rows))
    print("\n[2] Discrimination and calibration; C2 = AUROC >= 0.75 and lower 95% limit >= 0.70")
    rows = []
    for nm in ESC11:
        if nm not in results:
            continue
        pt, bt, y = results[nm]["point"], results[nm]["boot"], results[nm]["y"]
        for m in ("LR", "XGB", "Stack"):
            lo = np.nanpercentile(bt[(m, "auroc")], 2.5) if (m, "auroc") in bt else np.nan
            rows.append({"analysis": nm, "model": m, "events": int(y.sum()), "prevalence": f"{y.mean():.4f}",
                         "auroc": f"{pt[(m, 'auroc')]:.3f}" + _ci(bt, (m, "auroc")),
                         "auprc": f"{pt[(m, 'auprc')]:.3f}" + _ci(bt, (m, "auprc")),
                         "slope": f"{pt[(m, 'slope')]:.2f}", "O/E": f"{pt[(m, 'oe')]:.2f}",
                         "sens_top10": f"{pt[(m, 'sens_top10')]:.3f}",
                         "C2": "met" if (pt[(m, 'auroc')] >= 0.75 and lo >= 0.70) else "not met"})
    table(pd.DataFrame(rows))

if RUN_SECTION_11:
    emit("escalation_summary", escalation_summary)

escalation population: 19268  (expected 19268)  [OK]
escalation escalation: 101  (expected 101)  [OK]
escalamiento: admisiones=19268, pacientes=13632, eventos=101, numéricas=3, binarias=1, categóricas=5
escalation_labs: admisiones=19268, pacientes=13632, eventos=101, numéricas=14, binarias=12, categóricas=5
escalation_landmark: admisiones=19198, pacientes=13584, eventos=90, numéricas=3, binarias=1, categóricas=5


## E7 · E10 — raza y etnia: modelo sin la variable y desempeño por grupo

Las tasas de SMM difieren por raza y etnia, así que omitir la variable puede descalibrar el modelo en los grupos de mayor riesgo; pero la raza en los algoritmos clínicos también puede reproducir inequidades. `main_norace` y `landmark_norace` quitan `ethnicity`; `race_subgroups` reporta admisiones, eventos, AUROC, AUPRC, calibración global, pendiente y O/E por grupo (blanca, negra, hispana/latina, asiática, otra, desconocida o no declarada), con los conteos menores de 10 como `<10`.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Quitar la raza no cambia la discriminación pero empeora la calibración en algunos grupos | Mantener la raza en el modelo y reportar por grupo | Calibración peor en los grupos hispano y asiático sin raza |
| La discriminación de un grupo es claramente menor en la población landmark | Precaución explícita para cualquier implementación | Negras 0,689 frente a blancas 0,741 |

In [15]:
RACE_COL = "ethnicity"
MIN_EVENTS_GROUP = 20
SUB_MODELS = ("LR", "XGB", "Stack")

def race_group(v):
    s = str(v).upper().strip() if pd.notna(v) else "UNKNOWN"
    if s.startswith("WHITE"):
        return "White"
    if s.startswith("BLACK"):
        return "Black"
    if s.startswith("HISPANIC") or s.startswith("SOUTH AMERICAN"):
        return "Hispanic/Latina"
    if s.startswith("ASIAN"):
        return "Asian"
    if s in ("UNKNOWN", "UNABLE TO OBTAIN", "PATIENT DECLINED TO ANSWER", "NONE", ""):
        return "Unknown or declined"
    return "Other"

def _n(x):
    return str(int(x)) if x >= 10 else "<10"

if "_make_spec_s10" not in globals():   # wrap only once, so re-running the cell does not recurse
    _make_spec_s10 = make_spec
NORACE = ["main_norace", "landmark_norace"]

def make_spec(name):
    if name not in NORACE:
        return _make_spec_s10(name)
    sp = _make_spec_s10(name.replace("_norace", ""))
    assert RACE_COL in sp["cat"], f"{RACE_COL} not among the categorical predictors: {sp['cat']}"
    return dict(sp, name=name, cat=[c for c in sp["cat"] if c != RACE_COL])

# Mapping of the recorded categories to the collapsed groups (eligible deliveries; aggregates only)
e = df[df.eligible_24h & df.is_delivery_v2]
mp = (e.assign(group=e[RACE_COL].map(race_group))
        .groupby(["group", RACE_COL], dropna=False).agg(admissions=("y", "size"), smm=("y", "sum")).reset_index())
mp["admissions"] = mp["admissions"].map(_n); mp["smm"] = mp["smm"].map(_n)
print("Recorded race and ethnicity categories and the collapsed group (counts below 10 suppressed):")
table(mp)
for nm in NORACE:
    sp = make_spec(nm)
    print(f"{nm:18s} admissions={len(sp['data'])}  events={int(sp['data'][sp['label']].sum())}  categorical={sp['cat']}")

def group_metrics(res, B, seed):
    """Metrics by collapsed race group, averaged over repeats; patient bootstrap within each group."""
    d = res["spec"]["data"]; y = res["y"].astype(float); g = res["g"]
    grp = d[RACE_COL].map(race_group).values
    rows, rng = [], np.random.default_rng(seed)
    for gr in ["White", "Black", "Hispanic/Latina", "Asian", "Other", "Unknown or declined"]:
        mk = grp == gr
        n, ev = int(mk.sum()), int(y[mk].sum())
        if n == 0:
            continue
        base = {"analysis": res["spec"]["name"], "group": gr, "admissions": _n(n), "events": _n(ev)}
        if ev < MIN_EVENTS_GROUP or (n - ev) < MIN_EVENTS_GROUP:
            rows.append(dict(base, model="-", note=f"fewer than {MIN_EVENTS_GROUP} events or non-events: counts only"))
            continue
        yg, gg = y[mk], g[mk]
        ug, inv = np.unique(gg, return_inverse=True)
        point = {m: pd.DataFrame([metrics_one(yg, res["preds"][m][r][mk], np.ones(n)) for r in range(N_REPEATS)]).mean()
                 for m in SUB_MODELS}
        boot = {m: {"auroc": [], "oe": []} for m in SUB_MODELS}
        for b in range(B):
            w = np.bincount(rng.integers(0, len(ug), len(ug)), minlength=len(ug))[inv].astype(float)
            if np.sum(w * yg) < 1 or np.sum(w * (1 - yg)) < 1:
                continue
            for m in SUB_MODELS:
                a, o = [], []
                for r in range(N_REPEATS):
                    pr = res["preds"][m][r][mk]
                    a.append(roc_auc_score(yg, pr, sample_weight=w))
                    o.append(np.sum(w * yg) / np.sum(w * pr))
                boot[m]["auroc"].append(np.mean(a)); boot[m]["oe"].append(np.mean(o))
        for m in SUB_MODELS:
            pt = point[m]
            ci = lambda k, dd: " ({:.{d}f}-{:.{d}f})".format(*np.nanpercentile(boot[m][k], [2.5, 97.5]), d=dd)
            rows.append(dict(base, model=m, prevalence=f"{ev / n:.4f}",
                             auroc=f"{pt['auroc']:.3f}" + ci("auroc", 3), auprc=f"{pt['auprc']:.3f}",
                             citl=f"{pt['citl']:.2f}", slope=f"{pt['slope']:.2f}",
                             oe=f"{pt['oe']:.2f}" + ci("oe", 2), note=""))
    return rows

def paired_race_diff(res_a, res_b, B, seed):
    """Overall AUROC/AUPRC with race (a) minus without race (b), same patient bootstrap samples."""
    assert np.array_equal(res_a["y"], res_b["y"]) and np.array_equal(res_a["g"], res_b["g"]), "rows not aligned"
    y = res_a["y"].astype(float); ug, inv = np.unique(res_a["g"], return_inverse=True)
    rng = np.random.default_rng(seed); rows = []
    for m in SUB_MODELS:
        dif = {"auroc": [], "auprc": []}
        for b in range(B):
            w = np.bincount(rng.integers(0, len(ug), len(ug)), minlength=len(ug))[inv].astype(float)
            if np.sum(w * y) < 1:
                continue
            for met, fn in (("auroc", roc_auc_score), ("auprc", average_precision_score)):
                va = np.mean([fn(y, res_a["preds"][m][r], sample_weight=w) for r in range(N_REPEATS)])
                vb = np.mean([fn(y, res_b["preds"][m][r], sample_weight=w) for r in range(N_REPEATS)])
                dif[met].append(va - vb)
        r0 = {"comparison": f"{res_a['spec']['name']} - {res_b['spec']['name']}", "model": m}
        for met in ("auroc", "auprc"):
            pa = res_a["point"][(m, met)]; pb = res_b["point"][(m, met)]
            lo, hi = np.nanpercentile(dif[met], [2.5, 97.5])
            r0[met] = f"{pa - pb:+.3f} ({lo:+.3f} to {hi:+.3f})"
        rows.append(r0)
    return rows

RUN_SECTION_12 = True
if RUN_SECTION_12:
    for nm in ["main", "landmark"]:
        if nm not in results:
            results[nm] = run_analysis(nm, BOOT_SENS)
            emit(nm + "_rerun12", lambda r=results[nm]: report(r))
    for nm in NORACE:
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

def race_summary():
    print(f"RACE AND ETHNICITY SUMMARY (QUICK={QUICK}); bootstrap B={BOOT_SENS}; groups with < {MIN_EVENTS_GROUP} events or non-events: counts only")
    print("\n[1] Overall discrimination, with race minus without race (paired patient bootstrap)")
    rows = []
    for a, b in (("main", "main_norace"), ("landmark", "landmark_norace")):
        if a in results and b in results:
            rows += paired_race_diff(results[a], results[b], BOOT_SENS, SEED + 120)
    table(pd.DataFrame(rows))
    print("\n[2] Performance and calibration by racial and ethnic group")
    rows = []
    for nm in ("main", "main_norace", "landmark", "landmark_norace"):
        if nm in results:
            rows += group_metrics(results[nm], BOOT_SENS, SEED + 121)
    table(pd.DataFrame(rows).fillna(""))

if RUN_SECTION_12:
    emit("race_subgroups", race_summary)

Categorías de raza y etnia registradas y el grupo agrupado (conteos menores de 10 suprimidos):


grupo,ethnicity,admisiones,SMM
Asian,ASIAN,842,22
Asian,ASIAN - ASIAN INDIAN,127,<10
Asian,ASIAN - CHINESE,661,17
Asian,ASIAN - KOREAN,56,<10
Asian,ASIAN - SOUTH EAST ASIAN,106,<10
Black,BLACK/AFRICAN,244,<10
Black,BLACK/AFRICAN AMERICAN,3396,92
Black,BLACK/CAPE VERDEAN,624,10
Black,BLACK/CARIBBEAN ISLAND,180,<10
Hispanic/Latina,HISPANIC OR LATINO,545,<10


main_norace        admissions=19383  events=445  categorical=['insurance', 'language', 'admission_type', 'admission_location']
landmark_norace    admissions=19198  events=285  categorical=['insurance', 'language', 'admission_type', 'admission_location']
[main] el primer pliegue externo tardó 0.9 min -> total proyectado de unos 13 min
[main] validación cruzada terminada en 13.3 min; cada admisión evaluada exactamente una vez por repetición: Sí
  bootstrap: 5 réplicas tardaron 3 s -> proyectado 2.9 min para B=300


[reporte agregado guardado como data/outputs_main_rerun12.txt  (7829 caracteres)]
[landmark] el primer pliegue externo tardó 1.3 min -> total proyectado de unos 20 min
[landmark] validación cruzada terminada en 17.1 min; cada admisión evaluada exactamente una vez por repetición: Sí
  bootstrap: 5 réplicas tardaron 2 s -> proyectado 2.1 min para B=300


[reporte agregado guardado como data/outputs_landmark_rerun12.txt  (7785 caracteres)]
[main_norace] el primer pliegue externo tardó 0.8 min -> total proyectado de unos 12 min
[main_norace] validación cruzada terminada en 10.7 min; cada admisión evaluada exactamente una vez por repetición: Sí
  bootstrap: 5 réplicas tardaron 1 s -> proyectado 1.3 min para B=300


[reporte agregado guardado como data/outputs_main_norace.txt  (7402 caracteres)]
[landmark_norace] el primer pliegue externo tardó 0.9 min -> total proyectado de unos 14 min
[landmark_norace] validación cruzada terminada en 13.0 min; cada admisión evaluada exactamente una vez por repetición: Sí
  bootstrap: 5 réplicas tardaron 1 s -> proyectado 1.3 min para B=300


[reporte agregado guardado como data/outputs_landmark_norace.txt  (7753 caracteres)]


[reporte agregado guardado como data/outputs_race_subgroups.txt  (10413 caracteres)]


## R4 · C3 — sesgo de detección del modelo con laboratorios ampliados

La misma prueba de la sección 8 para el modelo del retorno R4. C3 cumple si el AUROC cae 0,05 o menos de `aki_lab_incident_labs` a `aki_lab_incident_tested_labs` (admisiones con creatinina medida en las primeras 24 h).

In [16]:
if "_make_spec_s12" not in globals():   # wrap only once, so re-running the cell does not recurse
    _make_spec_s12 = make_spec
C3LABS = ["aki_lab_incident_labs", "aki_lab_incident_tested_labs"]

def make_spec(name):
    if name not in C3LABS:
        return _make_spec_s12(name)
    sp = _make_spec_s12("aki_lab_incident")
    d = sp["data"]
    if name == "aki_lab_incident_tested_labs":
        d = d[d[CREAT].notna()]
    d = d.reset_index(drop=True)
    num = list(dict.fromkeys(sp["num"] + [v for v in VAR_LAB if v in d.columns]))
    binc = list(dict.fromkeys(sp["bin"] + [f"{v}_missing" for v in VAR_LAB if f"{v}_missing" in d.columns]))
    binc = [b for b in binc if b in d.columns and d[b].nunique() > 1]
    return dict(name=name, data=d, num=num, bin=binc, cat=sp["cat"], label=sp["label"])

for nm in C3LABS:
    sp = make_spec(nm)
    print(f"{nm:30s} admissions={len(sp['data'])}  events={int(sp['data'][sp['label']].sum())}  "
          f"numeric={len(sp['num'])}  binary={len(sp['bin'])}  categorical={len(sp['cat'])}")

RUN_SECTION_13 = True
if RUN_SECTION_13:
    for nm in C3LABS:
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

def c3_labs_summary():
    print(f"CRITERION C3 FOR THE EXTENDED LABORATORY MODEL (QUICK={QUICK}); bootstrap B={BOOT_SENS}")
    rows = []
    for m in ("LR", "XGB", "Stack"):
        r = {"model": m}
        for nm in C3LABS:
            if nm in results:
                pt, bt = results[nm]["point"], results[nm]["boot"]
                r[f"{nm} events"] = int(results[nm]["y"].sum())
                r[f"{nm} auroc"] = f"{pt[(m, 'auroc')]:.3f}" + _ci(bt, (m, "auroc"))
        if all(nm in results for nm in C3LABS):
            drop = results[C3LABS[0]]["point"][(m, "auroc")] - results[C3LABS[1]]["point"][(m, "auroc")]
            r["auroc drop"] = f"{drop:+.3f}"
            r["C3"] = "met" if drop <= 0.05 else "not met"
        rows.append(r)
    table(pd.DataFrame(rows))

if RUN_SECTION_13:
    emit("c3_labs_summary", c3_labs_summary)

aki_lab_incident_labs          admissions=19175  events=202  numeric=14  binary=12  categorical=5
aki_lab_incident_tested_labs   admissions=3767  events=165  numeric=14  binary=10  categorical=5
[aki_lab_incident_labs] el primer pliegue externo tardó 1.1 min -> total proyectado de unos 17 min
[aki_lab_incident_labs] validación cruzada terminada en 14.8 min; cada admisión evaluada exactamente una vez por repetición: Sí
  bootstrap: 5 réplicas tardaron 1 s -> proyectado 1.2 min para B=300


[reporte agregado guardado como data/outputs_aki_lab_incident_labs.txt  (7653 caracteres)]
[aki_lab_incident_tested_labs] el primer pliegue externo tardó 0.5 min -> total proyectado de unos 8 min
[aki_lab_incident_tested_labs] validación cruzada terminada en 8.1 min; cada admisión evaluada exactamente una vez por repetición: Sí
  bootstrap: 5 réplicas tardaron 1 s -> proyectado 0.5 min para B=300


[reporte agregado guardado como data/outputs_aki_lab_incident_tested_labs.txt  (7630 caracteres)]


modelo,aki_lab_incident_labs events,aki_lab_incident_labs auroc,aki_lab_incident_tested_labs events,aki_lab_incident_tested_labs auroc,caída del AUROC,C3
LR,202,0.854 (0.827-0.876),165,0.623 (0.580-0.670),+0.231,no cumple
XGB,202,0.861 (0.837-0.883),165,0.643 (0.601-0.685),+0.218,no cumple
Stack,202,0.860 (0.835-0.882),165,0.649 (0.608-0.689),+0.211,no cumple


[reporte agregado guardado como data/outputs_c3_labs_summary.txt  (698 caracteres)]


## R2 · P3 — tamaño de muestra para el desenlace del retorno R2 (escalamiento de la atención)

El retorno R2 devuelve el proceso a E3 con un desenlace nuevo, así que ese desenlace debe pasar otra vez la puerta P3. Criterios de Riley con el R² de Cox-Snell de la regresión logística con validación cruzada; se agrega el análisis principal de SMM como control (se requieren 9.112 admisiones y 209 eventos). Si las predicciones fuera de muestra no están en memoria, la validación cruzada se vuelve a correr sin bootstrap.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Eventos disponibles ≥ requeridos | P3 pasa | Modelo principal de SMM |
| Eventos disponibles < requeridos | Reducción por tamaño de muestra (sección 15) | Escalamiento: 101 eventos, se requieren 194 con 32 parámetros |

In [17]:
def riley(P, phi, r2cs, shrink=0.9, deltas=(0.05, 0.01)):
    lnl0 = phi * math.log(phi) + (1 - phi) * math.log(1 - phi)
    max_r2 = 1 - math.exp(2 * lnl0)
    n1 = P / ((shrink - 1) * math.log(1 - r2cs / shrink))
    s_vh = r2cs / (r2cs + 0.05 * max_r2)
    n2 = P / ((s_vh - 1) * math.log(1 - r2cs / s_vh))
    out = {"max_r2cs": max_r2, "n_shrinkage": n1, "n_optimism": n2}
    for dl in deltas:
        out[f"n_risk_delta{dl}"] = (1.96 / dl) ** 2 * phi * (1 - phi)
    n = max(n1, n2, out[f"n_risk_delta{deltas[-1]}"])
    out["n_required"] = n
    out["events_required"] = n * phi
    out["epp_at_required"] = n * phi / P
    out["P_max_supported"] = None
    return out

def p_max(n, r2cs, shrink=0.9):
    """Largest number of parameters with expected shrinkage >= 0.9 for a given n and R2 (criterion 1 solved for P)."""
    return n * (shrink - 1) * math.log(1 - r2cs / shrink)

def riley_block(label, n, events, P, r2_list):
    phi = events / n
    print(f"\n{label}: n={n}, events={events}, prevalence={phi:.5f}, candidate parameters P={P}, EPP={events/P:.2f}")
    rows = []
    for tag, r2 in r2_list:
        rr = riley(P, phi, r2)
        rows.append({"R2_CS scenario": tag, "R2_CS": f"{r2:.4f}", "max R2_CS": f"{rr['max_r2cs']:.4f}",
                     "n shrinkage": f"{rr['n_shrinkage']:.0f}", "n optimism": f"{rr['n_optimism']:.0f}",
                     "n risk(+-0.05)": f"{rr['n_risk_delta0.05']:.0f}", "n risk(+-0.01)": f"{rr['n_risk_delta0.01']:.0f}",
                     "n required": f"{rr['n_required']:.0f}", "events required": f"{rr['events_required']:.0f}",
                     "available n": n, "available events": events,
                     "max P supported (n,R2)": f"{p_max(n, r2):.1f}"})
    table(pd.DataFrame(rows))

def count_parameters(spec):
    d = spec["data"]
    X = prep_frame(d, spec["num"], spec["bin"], spec["cat"])
    return int(make_prep(spec["num"], spec["bin"], spec["cat"]).fit(X).transform(X).shape[1])

def r2cs_from_oof(res):
    y = res["y"].astype(float); n = len(y); phi = y.mean()
    lnl0 = n * (phi * math.log(phi) + (1 - phi) * math.log(1 - phi))
    vals = []
    for r in range(N_REPEATS):
        p = np.clip(res["preds"]["LR"][r], 1e-9, 1 - 1e-9)
        lnl1 = float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))
        vals.append(max(1 - math.exp(2 / n * (lnl0 - lnl1)), 1e-6))
    return float(np.mean(vals))
P3_ANALYSES = ["main", "escalation", "escalation_labs", "escalation_landmark"]

RUN_SECTION_14 = True
RUN_SECTION_14 = False

p3_res = {}
if RUN_SECTION_14:
    for nm in P3_ANALYSES:
        if nm in results:
            p3_res[nm] = results[nm]
        else:
            print(f"[{nm}] not in memory: re-running the cross-validation without bootstrap")
            p3_res[nm] = run_cv(make_spec(nm))

def p3_report():
    print(f"GATE G3 (RILEY CRITERIA) FOR THE OUTCOME OF RETURN R2 (QUICK={QUICK})")
    print("R2_CS observed = mean over repeats of the Cox-Snell R2 of the out-of-fold logistic regression predictions")
    for nm in P3_ANALYSES:
        if nm not in p3_res:
            continue
        res = p3_res[nm]; sp = res["spec"]
        n, ev = len(res["y"]), int(res["y"].sum())
        P = count_parameters(sp)
        r2 = r2cs_from_oof(res)
        riley_block(f"{nm} (label {sp['label']})", n, ev, P, [("observed (cross-validated LR)", r2)])
        verdict = "passes" if riley(P, ev / n, r2)["n_required"] <= n else "does NOT pass"
        print(f"  -> gate G3 {verdict}: {n} admissions available")

if RUN_SECTION_14:
    emit("g3_escalation", p3_report)


## P3 · reducción por tamaño de muestra para el desenlace del retorno R2

Regla registrada en `docs/decision_G3_reduction_2026-11-09.md` antes de correr esta sección: un modelo que cumple todos los criterios aplicables de P4 pero no pasa P3 se reajusta **una vez** con un conjunto reducido de predictores fijado de antemano (como máximo los parámetros que los datos sostienen, elegidos sin mirar el desenlace) y se revisa otra vez en P3 y en C1 y C2. Su C4 está en el notebook 09, sección 10.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| P3 pasa y se cumplen C1 y C2 | Señal exploratoria de predicción | Pasa P3 (se requieren 89 eventos); landmark 0,752; XGBoost 0,749 |
| P3 vuelve a fallar | Solo factibilidad | — |

In [18]:
def add_reduced_columns(frame):
    """Columns of the prespecified reduced set (docs/decision_G3_reduction_2026-11-09.md). Row-wise, no outcome information."""
    eth = frame["ethnicity"].fillna("").astype(str).str.upper()
    frame["race5"] = np.select(
        [eth.str.startswith("WHITE"), eth.str.startswith("BLACK"),
         eth.str.startswith("HISPANIC") | eth.str.startswith("SOUTH AMERICAN"), eth.str.startswith("ASIAN")],
        ["White", "Black", "Hispanic or Latina", "Asian"], default="Other or unknown")
    frame["ins_medicaid"] = (frame["insurance"].astype(str) == "Medicaid").astype(int)
    frame["lang_english"] = (frame["language"].astype(str) == "English").astype(int)
    frame["adm_urgent"] = (frame["admission_type"].astype(str) == "URGENT").astype(int)
    frame["loc_physician"] = (frame["admission_location"].astype(str) == "PHYSICIAN REFERRAL").astype(int)
    return frame

RED_BIN = ["ins_medicaid", "lang_english", "adm_urgent", "loc_physician"]
RED_CAT = ["race5"]
P_MAX_REDUCTION = 16

df = add_reduced_columns(df)
if "_make_spec_s15" not in globals():   # wrap only once, so re-running the cell does not recurse
    _make_spec_s15 = make_spec
RED15 = ["escalation_reduced", "escalation_landmark_reduced"]

def make_spec(name):
    if name not in RED15:
        return _make_spec_s15(name)
    sp = _make_spec_s15(name.replace("_reduced", ""))
    d = add_reduced_columns(sp["data"].copy())
    binc = [b for b in (["creatinine_missing"] + RED_BIN) if b in d.columns and d[b].nunique() > 1]
    return dict(name=name, data=d, num=list(sp["num"]), bin=binc, cat=list(RED_CAT), label=translate("y_esc", LANG))

P_RED = count_parameters(make_spec("escalation_reduced"))
print(f"Reduced set: {P_RED} parameters (maximum allowed {P_MAX_REDUCTION})")
assert P_RED <= P_MAX_REDUCTION, "reduced set exceeds the parameters supported by the data: STOP (do not modify after seeing results)"

RUN_SECTION_15 = True
if RUN_SECTION_15:
    for nm in RED15:
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

def reduced_summary():
    print(f"SAMPLE-SIZE REDUCTION (G3) FOR ESCALATION (RETURN R2), QUICK={QUICK}, bootstrap B={BOOT_SENS}")
    print("Rule and reduced set: docs/decision_G3_reduction_2026-11-09.md (fixed before this run)")
    full, lm = results["escalation_reduced"], results["escalation_landmark_reduced"]
    n, ev = len(full["y"]), int(full["y"].sum())
    r2 = r2cs_from_oof(full)
    riley_block("escalation_reduced (label y_esc)", n, ev, P_RED, [("observed (cross-validated LR, reduced set)", r2)])
    p3_ok = riley(P_RED, ev / n, r2)["n_required"] <= n
    print(f"  -> gate G3 {'passes' if p3_ok else 'does NOT pass'}")
    rows = []
    for m in ("LR", "XGB", "Stack"):
        pf, pl = full["point"], lm["point"]
        lo = np.nanpercentile(full["boot"][(m, "auroc")], 2.5)
        prev_f, prev_l = full["y"].mean(), lm["y"].mean()
        gain_f, gain_l = pf[(m, "auprc")] / prev_f, pl[(m, "auprc")] / prev_l
        kept = (gain_l - 1) / (gain_f - 1) if gain_f > 1 else np.nan
        drop = pf[(m, "auroc")] - pl[(m, "auroc")]
        rows.append({"model": m, "events": ev,
                     "auroc": f"{pf[(m, 'auroc')]:.3f}" + _ci(full["boot"], (m, "auroc")),
                     "auprc": f"{pf[(m, 'auprc')]:.3f}" + _ci(full["boot"], (m, "auprc")),
                     "slope": f"{pf[(m, 'slope')]:.2f}", "O/E": f"{pf[(m, 'oe')]:.2f}",
                     "sens_top10": f"{pf[(m, 'sens_top10')]:.3f}",
                     "C2": "met" if (pf[(m, "auroc")] >= 0.75 and lo >= 0.70) else "not met",
                     "landmark auroc": f"{pl[(m, 'auroc')]:.3f}" + _ci(lm["boot"], (m, "auroc")),
                     "auroc drop": f"{drop:.3f}", "AUPRC gain kept": f"{kept:.2f}",
                     "C1": "met" if (drop <= 0.05 and kept >= 2 / 3) else "not met"})
    table(pd.DataFrame(rows))
    print("C3: not applicable. C4: notebook 09, section 10.")

if RUN_SECTION_15:
    emit("escalation_reduced_summary", reduced_summary)

Conjunto reducido: 13 parámetros (máximo permitido 16)
[escalation_reduced] el primer pliegue externo tardó 0.8 min -> total proyectado de unos 12 min
[escalation_reduced] validación cruzada terminada en 11.2 min; cada admisión evaluada exactamente una vez por repetición: Sí
  bootstrap: 5 réplicas tardaron 1 s -> proyectado 1.3 min para B=300


[reporte agregado guardado como data/outputs_escalation_reduced.txt  (7711 caracteres)]
[escalation_landmark_reduced] el primer pliegue externo tardó 0.8 min -> total proyectado de unos 12 min
[escalation_landmark_reduced] validación cruzada terminada en 11.3 min; cada admisión evaluada exactamente una vez por repetición: Sí
  bootstrap: 5 réplicas tardaron 1 s -> proyectado 1.2 min para B=300


[reporte agregado guardado como data/outputs_escalation_landmark_reduced.txt  (7603 caracteres)]


escenario de R2_CS,R2_CS,R2_CS máx.,n encogimiento,n optimismo,n riesgo(±-0.05),n riesgo(±-0.01),n requerido,eventos requeridos,n disponible,eventos disponibles,"P máx. soportado (n, R2)"
"observed (cross-validated LR, reduced set)",0.0069,0.0634,16962,4080,8,200,16962,89,"19,268",101,14.8
modelo,eventos,AUROC,AUPRC,pendiente,O/E,sens. 0% superior,C2,AUROC landmark,caída del AUROC,ganancia del AUPRC conservada,C1
LR,101,0.764 (0.707-0.812),0.041 (0.026-0.072),1.07,1.00,0.485,cumple,0.761 (0.711-0.815),0.003,1.03,cumple
XGB,101,0.749 (0.692-0.798),0.036 (0.025-0.064),1.05,0.97,0.439,no cumple,0.754 (0.699-0.811),-0.006,1.19,cumple
Stack,101,0.763 (0.703-0.811),0.039 (0.025-0.066),1.08,0.99,0.502,cumple,0.769 (0.718-0.825),-0.006,1.07,cumple


[reporte agregado guardado como data/outputs_escalation_reduced_summary.txt  (1280 caracteres)]


## Exploratorio (post hoc) — malla de XGBoost ampliada para el modelo de escalamiento reducido

Decidido **después** de ver la sección 15 (AUROC de XGBoost 0,749, justo por debajo de C2). La malla de XGBoost se amplía a 36 configuraciones; todo lo demás no cambia y la configuración se sigue eligiendo solo dentro de los pliegues internos. Se reporta como exploratorio.

In [19]:
GRID_XGB_SMALL = list(GRID["XGB"])
GRID_XGB_WIDE = [dict(depth=dp, spw=sp, lr=lr, mcw=mcw)
                 for dp in (1, 2, 3) for sp in (1, "sqrt") for lr in (0.02, 0.05, 0.1) for mcw in (5, 20)]
_make_clf_base = globals().get("_make_clf_base", make_clf)

def make_clf(name, cfg, ytr, seed):
    if name != "XGB" or "lr" not in cfg:
        return _make_clf_base(name, cfg, ytr, seed)
    pos = max(int(ytr.sum()), 1); neg = max(len(ytr) - pos, 1)
    spw = math.sqrt(neg / pos) if cfg["spw"] == "sqrt" else 1.0
    return XGBClassifier(n_estimators=XGB_TREES, learning_rate=cfg["lr"], max_depth=cfg["depth"], subsample=0.8,
                         colsample_bytree=0.8, min_child_weight=cfg["mcw"], scale_pos_weight=spw, eval_metric="logloss",
                         tree_method="hist", n_jobs=N_JOBS if N_JOBS and N_JOBS > 0 else 0, random_state=seed)

RUN_SECTION_16 = True
if RUN_SECTION_16:
    GRID["XGB"] = GRID_XGB_WIDE
    try:
        for nm in RED15:
            key = nm + "_xgbgrid"
            res = run_analysis(nm, BOOT_SENS)
            res["spec"] = dict(res["spec"], name=key)
            results[key] = res
            emit(key, lambda r=res: report(r))
    finally:
        GRID["XGB"] = GRID_XGB_SMALL   # restore the prespecified grid for any later section

def xgbgrid_summary():
    print(f"EXPLORATORY (post hoc): WIDER XGBOOST GRID, REDUCED ESCALATION MODEL, QUICK={QUICK}, bootstrap B={BOOT_SENS}")
    print(f"XGBoost grid: {len(GRID_XGB_WIDE)} configurations (prespecified grid: {len(GRID_XGB_SMALL)}); tuned inside inner folds only")
    full, lm = results["escalation_reduced_xgbgrid"], results["escalation_landmark_reduced_xgbgrid"]
    rows = []
    for m in ("LR", "XGB", "Stack"):
        pf, pl = full["point"], lm["point"]
        lo = np.nanpercentile(full["boot"][(m, "auroc")], 2.5)
        gain_f, gain_l = pf[(m, "auprc")] / full["y"].mean(), pl[(m, "auprc")] / lm["y"].mean()
        kept = (gain_l - 1) / (gain_f - 1) if gain_f > 1 else np.nan
        drop = pf[(m, "auroc")] - pl[(m, "auroc")]
        rows.append({"model": m, "auroc": f"{pf[(m, 'auroc')]:.3f}" + _ci(full["boot"], (m, "auroc")),
                     "auprc": f"{pf[(m, 'auprc')]:.3f}", "slope": f"{pf[(m, 'slope')]:.2f}", "O/E": f"{pf[(m, 'oe')]:.2f}",
                     "C2": "met" if (pf[(m, "auroc")] >= 0.75 and lo >= 0.70) else "not met",
                     "landmark auroc": f"{pl[(m, 'auroc')]:.3f}", "auroc drop": f"{drop:.3f}",
                     "gain kept": f"{kept:.2f}", "C1": "met" if (drop <= 0.05 and kept >= 2 / 3) else "not met"})
    table(pd.DataFrame(rows))
    cnt = pd.Series([c for n, c in full["chosen"] if n == "XGB"]).value_counts()
    print("\nXGBoost configurations chosen in the inner loops (full population):")
    table(cnt, index=True)

if RUN_SECTION_16:
    emit("escalation_reduced_xgbgrid_summary", xgbgrid_summary)

[escalation_reduced] el primer pliegue externo tardó 1.1 min -> total proyectado de unos 16 min
[escalation_reduced] validación cruzada terminada en 15.3 min; cada admisión evaluada exactamente una vez por repetición: Sí
  bootstrap: 5 réplicas tardaron 1 s -> proyectado 1.1 min para B=300


[reporte agregado guardado como data/outputs_escalation_reduced_xgbgrid.txt  (8099 caracteres)]
[escalation_landmark_reduced] el primer pliegue externo tardó 1.1 min -> total proyectado de unos 17 min
[escalation_landmark_reduced] validación cruzada terminada en 15.4 min; cada admisión evaluada exactamente una vez por repetición: Sí
  bootstrap: 5 réplicas tardaron 1 s -> proyectado 1.1 min para B=300


[reporte agregado guardado como data/outputs_escalation_landmark_reduced_xgbgrid.txt  (8049 caracteres)]


[reporte agregado guardado como data/outputs_escalation_reduced_xgbgrid_summary.txt  (1047 caracteres)]


## Resumen del notebook

Criterios C1 a C3 con modelos reajustados, y retornos R4 y R2.

In [20]:
# Summary of notebook 07
_rows = []
for _nm, _lab in [("landmark", "C1 landmark"), ("incident_documented", "C2 documented incident"),
                  ("incident_strict", "C2 strict incident"), ("aki_lab_incident", "C3 laboratory AKI, incident"),
                  ("labs_extended", "R4 extended laboratories"), ("escalation", "R2 escalation of care"),
                  ("escalation_reduced", "R2 escalation, reduced set (G3)")]:
    _rows.append((f"AUROC XGBoost, {_lab} (95% CI)", lambda nm=_nm: _m(results[nm], "XGB", "auroc")))
_rows.append(("classification at gate G4", "restated on the fitted models in notebook 10"))
notebook_summary(_rows)

,valor
"AUROC XGBoost, C1 landmark (IC 95%)",0.723 (0.690-0.753)
"AUROC XGBoost, C2 incidente documentada (IC 95%)",n/d
"AUROC XGBoost, C2 incidente estricta (IC 95%)",n/d
"AUROC XGBoost, C3 lesión renal aguda por laboratorio, incidente (IC 95%)",n/d
"AUROC XGBoost, R4 laboratorios ampliados (IC 95%)",n/d
"AUROC XGBoost, R2 escalamiento de la atención (IC 95%)",n/d
"AUROC XGBoost, R2 escalamiento, conjunto reducido (P3) (IC 95%)",0.749 (0.692-0.798)
clasificación en la puerta P4,se establece sobre los modelos ajustados en el notebook 10
